In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/latest.pt
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/log.jsonl
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt


In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Read token securely from Kaggle Secrets
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

REPO_ID = "Exploration-Lab/CS-781-Capstone"
LOCAL_DIR = "/kaggle/working/Phase_3_RR"

# Download only Phase 3 files
path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=hf_token,
    allow_patterns=[
        "Phase_3_RR/**"
    ],
    local_dir=LOCAL_DIR,
)

print("Download completed!")
print("Dataset location:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Download completed!
Dataset location: /kaggle/working/Phase_3_RR


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/working/Phase_3_RR"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/working/Phase_3_RR/Phase_3_RR/Train/IT_train.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/CL_train.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/IT_dev.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/CL_dev.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/label_vocab.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Test/IT_test.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Test/CL_test.json
/kaggle/working/Phase_3_RR/.cache/huggingface/CACHEDIR.TAG
/kaggle/working/Phase_3_RR/.cache/huggingface/.gitignore
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Train/IT_dev.json.metadata
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Train/CL_train.json.lock
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Train/IT_train.json.lock
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Train/IT_dev.json.lock
/kaggle/working/Phase_3_RR/.cache/huggingface/download/Phase_3_RR/Train/IT_train.json.metadata
/kaggle/working/Phase_3_

In [4]:
import os
import json
from collections import Counter

DATA_DIR = "/kaggle/working/Phase_3_RR/Phase_3_RR"
TRAIN_DIR = os.path.join(DATA_DIR, "Train")
TEST_DIR = os.path.join(DATA_DIR, "Test")

# Load official label vocabulary
with open(os.path.join(TRAIN_DIR, "label_vocab.json"), "r") as f:
    label_vocab = json.load(f)

print("Official label vocabulary:")
print(label_vocab)

def load_json(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return json.load(f)

for filename in [
    "CL_train.json",
    "CL_dev.json",
    "IT_train.json",
    "IT_dev.json",
]:
    path = os.path.join(TRAIN_DIR, filename)
    data = load_json(path)

    print(f"\n{'=' * 50}")
    print(filename)
    print("Data type:", type(data).__name__)

    # Support both list-of-documents and dict-of-documents formats
    documents = (
        list(data.values())
        if isinstance(data, dict) and "text" not in data
        else data
    )

    print("Documents:", len(documents))

    if not documents:
        continue

    first_doc = documents[0]
    print("First document keys:", list(first_doc.keys()))

    if isinstance(first_doc.get("text"), list):
        print("Sentences in first document:", len(first_doc["text"]))

    if isinstance(first_doc.get("labels"), list):
        print("Labels in first document:", len(first_doc["labels"]))

    if isinstance(first_doc.get("labels"), list):
        all_labels = [
            label
            for doc in documents
            for label in doc.get("labels", [])
        ]

        print("Total labeled sentences:", len(all_labels))
        print("Label distribution:", Counter(all_labels))

Official label vocabulary:
{'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}

CL_train.json
Data type: list
Documents: 40
First document keys: ['id', 'text', 'labels']
Sentences in first document: 43
Labels in first document: 43
Total labeled sentences: 10593
Label distribution: Counter({'Fact': 2979, 'RatioOfTheDecision': 2856, 'PrecedentReliedUpon': 1305, 'ArgumentRespondent': 996, 'ArgumentPetitioner': 976, 'RulingByLowerCourt': 438, 'Statute': 367, 'Dissent': 284, 'PrecedentNotReliedUpon': 209, 'RulingByPresentCourt': 113, 'None': 53, 'Issue': 17})

CL_dev.json
Data type: list
Documents: 5
First document keys: ['id', 'text', 'labels']
Sentences in first document: 265
Labels in first document: 265
Total labeled sentences: 1396
Label distribution: Counter({'Argumen

In [5]:
import os

SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/working",
]

seen = set()

for base in SEARCH_ROOTS:
    for root, dirs, files in os.walk(base):
        dirs[:] = [
            d for d in dirs
            if d not in {".cache", "__pycache__"}
        ]

        for file in files:
            if file.endswith((".pt", ".pth", ".bin", ".safetensors")):
                path = os.path.join(root, file)

                if path not in seen:
                    seen.add(path)
                    print(path)

/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/latest.pt
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt


In [6]:
import torch
import os

CKPT_PATH = (
    "/kaggle/input/models/achiver7/"
    "arch1-modeldecoder/transformers/default/1/"
    "run_arch1_real/best.pt"
)

print("Checkpoint exists:", os.path.exists(CKPT_PATH))
print("Checkpoint size (MB):", round(os.path.getsize(CKPT_PATH) / 1024**2, 2))

checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
        elif isinstance(value, torch.Tensor):
            print(f"{key}: Tensor {tuple(value.shape)}")
        else:
            print(f"{key}: {value}")

Checkpoint exists: True
Checkpoint size (MB): 1041.62

Checkpoint type: dict

Top-level keys:
model: dictionary with 141 entries
opt: dictionary with 2 entries
scaler: dictionary with 5 entries
step: 10000
best: 4.100625859101613
cfg: dictionary with 12 entries
n_params: 88210944


In [7]:
import os

roots = [
    "/kaggle/input",
    "/kaggle/working",
]

for base in roots:
    for root, dirs, files in os.walk(base):
        dirs[:] = [
            d for d in dirs
            if d not in {".cache", "__pycache__"}
        ]

        for file in files:
            if file.endswith(".py"):
                path = os.path.join(root, file)

                if any(
                    term in file.lower()
                    for term in ["model", "arch1", "train", "config"]
                ):
                    print(path)

In [8]:
import os

for base in ["/kaggle/input", "/kaggle/working"]:
    for root, dirs, files in os.walk(base):
        dirs[:] = [
            d for d in dirs
            if d not in {".cache", "__pycache__"}
        ]

        for file in files:
            if file.endswith(".py"):
                path = os.path.join(root, file)

                if any(
                    word in file.lower()
                    for word in ["model", "arch1", "train", "config"]
                ):
                    print(path)

In [9]:
import torch

CKPT_PATH = (
    "/kaggle/input/models/achiver7/"
    "arch1-modeldecoder/transformers/default/1/"
    "run_arch1_real/best.pt"
)

checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

print("ARCH1 CONFIG")
for key, value in checkpoint["cfg"].items():
    print(f"{key}: {value}")

print("\nMODEL STATE DICT KEYS")
for key, value in checkpoint["model"].items():
    shape = tuple(value.shape) if isinstance(value, torch.Tensor) else type(value)
    print(key, shape)

ARCH1 CONFIG
BLOCK: 1024
MICRO_BATCH: 1
GRAD_ACCUM: 16
LR: 0.0003
MIN_LR_FRAC: 0.1
WD: 0.1
WARMUP: 500
STEPS: 10000
CLIP: 1.0
EVAL_EVERY: 500
SEED: 42
MIX: {'acts': 0.273, 'sc': 0.469, 'tri': 0.258}

MODEL STATE DICT KEYS
token_emb.weight (40000, 768)
pos_emb.weight (1024, 768)
blocks.0.norm1.weight (768,)
blocks.0.norm1.bias (768,)
blocks.0.attn.mask (1, 1, 1024, 1024)
blocks.0.attn.q_proj.weight (768, 768)
blocks.0.attn.q_proj.bias (768,)
blocks.0.attn.k_proj.weight (768, 768)
blocks.0.attn.k_proj.bias (768,)
blocks.0.attn.v_proj.weight (768, 768)
blocks.0.attn.v_proj.bias (768,)
blocks.0.attn.out_proj.weight (768, 768)
blocks.0.attn.out_proj.bias (768,)
blocks.0.norm2.weight (768,)
blocks.0.norm2.bias (768,)
blocks.0.mlp.fc1.weight (3072, 768)
blocks.0.mlp.fc1.bias (3072,)
blocks.0.mlp.fc2.weight (768, 3072)
blocks.0.mlp.fc2.bias (768,)
blocks.1.norm1.weight (768,)
blocks.1.norm1.bias (768,)
blocks.1.attn.mask (1, 1, 1024, 1024)
blocks.1.attn.q_proj.weight (768, 768)
blocks.1.attn.q

In [10]:
import os

keywords = [
    "class Arch1",
    "class Block",
    "class CausalSelfAttention",
    "class Transformer",
    "build_arch1",
    "token_emb",
]

for base in ["/kaggle/input", "/kaggle/working"]:
    for root, dirs, files in os.walk(base):
        dirs[:] = [
            d for d in dirs
            if d not in {".cache", "__pycache__"}
        ]

        for file in files:
            if not file.endswith(".py"):
                continue

            path = os.path.join(root, file)

            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    source = f.read()
            except OSError:
                continue

            matched = [k for k in keywords if k in source]

            if matched:
                print(f"\nFile: {path}")
                print("Matches:", matched)

In [11]:
import torch

CKPT_PATH = (
    "/kaggle/input/models/achiver7/"
    "arch1-modeldecoder/transformers/default/1/"
    "run_arch1_real/best.pt"
)

ckpt = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

sd = ckpt["model"]

print("=== ARCH1 STRUCTURE ===")

print("Transformer blocks:", len({
    key.split(".")[1]
    for key in sd
    if key.startswith("blocks.") and key.split(".")[1].isdigit()
}))

print("\n=== FIRST BLOCK KEYS ===")

for key, value in sd.items():
    if key.startswith("blocks.0."):
        print(key, tuple(value.shape) if isinstance(value, torch.Tensor) else type(value))

print("\n=== FINAL NORMALIZATION ===")

for key in sd:
    if key.startswith("final_norm."):
        print(key, tuple(sd[key].shape))

print("\n=== EMBEDDING WEIGHTS ===")

print("Token embedding:", tuple(sd["token_emb.weight"].shape))
print("LM head:", tuple(sd["lm_head.weight"].shape))
print("Values identical:", torch.equal(
    sd["token_emb.weight"],
    sd["lm_head.weight"]
))

print("\n=== MODEL CONFIG ===")

for key, value in ckpt["cfg"].items():
    print(f"{key}: {value}")

=== ARCH1 STRUCTURE ===
Transformer blocks: 8

=== FIRST BLOCK KEYS ===
blocks.0.norm1.weight (768,)
blocks.0.norm1.bias (768,)
blocks.0.attn.mask (1, 1, 1024, 1024)
blocks.0.attn.q_proj.weight (768, 768)
blocks.0.attn.q_proj.bias (768,)
blocks.0.attn.k_proj.weight (768, 768)
blocks.0.attn.k_proj.bias (768,)
blocks.0.attn.v_proj.weight (768, 768)
blocks.0.attn.v_proj.bias (768,)
blocks.0.attn.out_proj.weight (768, 768)
blocks.0.attn.out_proj.bias (768,)
blocks.0.norm2.weight (768,)
blocks.0.norm2.bias (768,)
blocks.0.mlp.fc1.weight (3072, 768)
blocks.0.mlp.fc1.bias (3072,)
blocks.0.mlp.fc2.weight (768, 3072)
blocks.0.mlp.fc2.bias (768,)

=== FINAL NORMALIZATION ===
final_norm.weight (768,)
final_norm.bias (768,)

=== EMBEDDING WEIGHTS ===
Token embedding: (40000, 768)
LM head: (40000, 768)
Values identical: True

=== MODEL CONFIG ===
BLOCK: 1024
MICRO_BATCH: 1
GRAD_ACCUM: 16
LR: 0.0003
MIN_LR_FRAC: 0.1
WD: 0.1
WARMUP: 500
STEPS: 10000
CLIP: 1.0
EVAL_EVERY: 500
SEED: 42
MIX: {'acts': 0.

In [12]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# CONFIGURATION
# ============================================================

VOCAB_SIZE = 40000
BLOCK_SIZE = 1024
D_MODEL = 768
N_HEADS = 12
N_LAYERS = 8
D_FF = 3072

CKPT_PATH = (
    "/kaggle/input/models/achiver7/"
    "arch1-modeldecoder/transformers/default/1/"
    "run_arch1_real/best.pt"
)

# ============================================================
# CAUSAL SELF-ATTENTION
# ============================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()

        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(
            torch.ones(block_size, block_size, dtype=torch.bool)
        )
        self.register_buffer(
            "mask",
            mask.view(1, 1, block_size, block_size),
            persistent=True,
        )

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(
            ~self.mask[:, :, :T, :T],
            torch.finfo(scores.dtype).min,
        )

        attn = F.softmax(scores, dim=-1)
        out = attn @ v

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)


# ============================================================
# TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(
            d_model, n_heads, block_size
        )

        self.norm2 = nn.LayerNorm(d_model)

        self.mlp = nn.ModuleDict({
            "fc1": nn.Linear(d_model, d_ff),
            "fc2": nn.Linear(d_ff, d_model),
        })

    def forward(self, x):
        x = x + self.attn(self.norm1(x))

        h = self.norm2(x)
        h = self.mlp["fc1"](h)
        h = F.gelu(h)
        h = self.mlp["fc2"](h)

        return x + h


# ============================================================
# ARCH1
# ============================================================

class Arch1(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        block_size=BLOCK_SIZE,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        d_ff=D_FF,
    ):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model, n_heads, d_ff, block_size
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Match the tied embedding weights in the checkpoint.
        self.lm_head.weight = self.token_emb.weight

    def forward(self, input_ids, return_hidden=False):
        B, T = input_ids.shape

        if T > BLOCK_SIZE:
            raise ValueError(
                f"Input length {T} exceeds {BLOCK_SIZE}"
            )

        positions = torch.arange(
            T, device=input_ids.device
        ).unsqueeze(0)

        x = self.token_emb(input_ids) + self.pos_emb(positions)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        if return_hidden:
            return x

        return self.lm_head(x)


# ============================================================
# STRICT CHECKPOINT VALIDATION
# ============================================================

checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False,
)

model = Arch1()

result = model.load_state_dict(
    checkpoint["model"],
    strict=True,
)

print("Strict checkpoint loading:", result)
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Expected parameters:", checkpoint["n_params"])
print("Parameter count matches:",
      sum(p.numel() for p in model.parameters()) == checkpoint["n_params"])

print("Embedding weights tied:",
      model.token_emb.weight is model.lm_head.weight)

# Basic forward-pass test
model.eval()

with torch.no_grad():
    dummy = torch.randint(0, VOCAB_SIZE, (1, 16))
    hidden = model(dummy, return_hidden=True)

print("Forward pass successful:", hidden.shape)

Strict checkpoint loading: <All keys matched successfully>
Model parameters: 88210944
Expected parameters: 88210944
Parameter count matches: True
Embedding weights tied: True
Forward pass successful: torch.Size([1, 16, 768])


In [13]:
import os

for base in ["/kaggle/input", "/kaggle/working"]:
    for root, dirs, files in os.walk(base):
        dirs[:] = [
            d for d in dirs
            if d not in {".cache", "__pycache__"}
        ]

        for file in files:
            name = file.lower()

            if any(
                key in name
                for key in [
                    "tokenizer",
                    "vocab",
                    "merges",
                    "token",
                    "spiece",
                ]
            ):
                print(os.path.join(root, file))

/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/working/Phase_3_RR/Phase_3_RR/Train/label_vocab.json


In [14]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [15]:
from tokenizers import Tokenizer

TOKENIZER_PATH = (
    "/kaggle/input/datasets/achiver07/"
    "phase1-tokens-v40k-clean1/tokenizer.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer loaded successfully")
print("Vocabulary size:", tokenizer.get_vocab_size())
print("Special tokens:", tokenizer.get_added_tokens_decoder())

for text in [
    "The petitioner filed the present appeal.",
    "The court held that the appeal must be dismissed.",
]:
    encoded = tokenizer.encode(text)
    print("\nText:", text)
    print("Token IDs:", encoded.ids)
    print("Decoded:", tokenizer.decode(encoded.ids))

Tokenizer loaded successfully
Vocabulary size: 40000
Special tokens: {0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 1: AddedToken("<bos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 2: AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)}

Text: The petitioner filed the present appeal.
Token IDs: [1410, 1961, 1049, 263, 1061, 805, 16]
Decoded: The petitioner filed the present appeal.

Text: The court held that the appeal must be dismissed.
Token IDs: [1410, 927, 893, 334, 263, 805, 1222, 300, 1976, 16]
Decoded: The court held that the appeal must be dismissed.


In [16]:
import torch
import tokenizers
import sklearn
import numpy

print("PyTorch:", torch.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
Tokenizers: 0.23.1
Scikit-learn: 1.6.1
NumPy: 2.1.3
GPU available: True


In [17]:
from pathlib import Path
import torch
from tokenizers import Tokenizer

# Paths
DATA_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")

TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR = DATA_DIR / "Test"

CHECKPOINT_PATH = Path(
    "/kaggle/input/models/achiver7/"
    "arch1-modeldecoder/transformers/default/1/"
    "run_arch1_real/best.pt"
)

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/achiver07/"
    "phase1-tokens-v40k-clean1/tokenizer.json"
)

# Verify files
required_files = [
    TRAIN_DIR / "CL_train.json",
    TRAIN_DIR / "CL_dev.json",
    TRAIN_DIR / "IT_train.json",
    TRAIN_DIR / "IT_dev.json",
    TRAIN_DIR / "label_vocab.json",
]

for path in required_files:
    print(f"{'OK' if path.exists() else 'MISSING'}: {path}")

print(f"\nCheckpoint exists: {CHECKPOINT_PATH.exists()}")
print(f"Tokenizer exists: {TOKENIZER_PATH.exists()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Verify special tokens
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("\nVocabulary size:", tokenizer.get_vocab_size())
print("PAD ID: 0 | BOS ID: 1 | EOS ID: 2")

test_text = "The court held that the appeal must be dismissed."
encoded = tokenizer.encode(test_text)

print("\nTokenizer test:")
print("Input:", test_text)
print("Token IDs:", encoded.ids)
print("Decoded:", tokenizer.decode(encoded.ids))

OK: /kaggle/working/Phase_3_RR/Phase_3_RR/Train/CL_train.json
OK: /kaggle/working/Phase_3_RR/Phase_3_RR/Train/CL_dev.json
OK: /kaggle/working/Phase_3_RR/Phase_3_RR/Train/IT_train.json
OK: /kaggle/working/Phase_3_RR/Phase_3_RR/Train/IT_dev.json
OK: /kaggle/working/Phase_3_RR/Phase_3_RR/Train/label_vocab.json

Checkpoint exists: True
Tokenizer exists: True
GPU: Tesla T4

Vocabulary size: 40000
PAD ID: 0 | BOS ID: 1 | EOS ID: 2

Tokenizer test:
Input: The court held that the appeal must be dismissed.
Token IDs: [1410, 927, 893, 334, 263, 805, 1222, 300, 1976, 16]
Decoded: The court held that the appeal must be dismissed.


In [18]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from types import SimpleNamespace

# ============================================================
# CONFIGURATION
# ============================================================

CHECKPOINT_PATH = Path(
    "/kaggle/input/models/achiver7/arch1-modeldecoder/"
    "transformers/default/1/run_arch1_real/best.pt"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 40000
BLOCK_SIZE = 1024
D_MODEL = 768
N_HEADS = 12
N_LAYERS = 8
D_FF = 3072

EXPECTED_PARAMS = 88_210_944

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

# ============================================================
# CAUSAL SELF-ATTENTION
# ============================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()

        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(
            torch.ones(block_size, block_size)
        ).view(1, 1, block_size, block_size)

        self.register_buffer("mask", mask)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        def split_heads(tensor):
            return tensor.view(
                batch_size,
                seq_len,
                self.n_heads,
                self.head_dim
            ).transpose(1, 2)

        q = split_heads(self.q_proj(x))
        k = split_heads(self.k_proj(x))
        v = split_heads(self.v_proj(x))

        scores = (
            q @ k.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        scores = scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len] == 0,
            float("-inf")
        )

        attention = F.softmax(scores, dim=-1)

        output = attention @ v

        output = (
            output.transpose(1, 2)
            .contiguous()
            .view(batch_size, seq_len, d_model)
        )

        return self.out_proj(output)


# ============================================================
# TRANSFORMER BLOCK
# IMPORTANT: MLP NAMES MATCH THE CHECKPOINT
# ============================================================

class MLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)

        self.attn = CausalSelfAttention(
            d_model, n_heads, block_size
        )

        self.norm2 = nn.LayerNorm(d_model)

        # Checkpoint keys:
        # blocks.0.mlp.fc1.weight
        # blocks.0.mlp.fc2.weight
        self.mlp = MLP(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# ============================================================
# ARCH1 LANGUAGE MODEL
# ============================================================

class Arch1(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        block_size=BLOCK_SIZE,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        d_ff=D_FF
    ):
        super().__init__()

        self.block_size = block_size

        self.token_emb = nn.Embedding(
            vocab_size, d_model
        )

        self.pos_emb = nn.Embedding(
            block_size, d_model
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                n_heads,
                d_ff,
                block_size
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(
            d_model, vocab_size, bias=False
        )

        # Preserve tied embeddings.
        self.lm_head.weight = self.token_emb.weight

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        if seq_len > self.block_size:
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"block size {self.block_size}"
            )

        positions = torch.arange(
            seq_len, device=input_ids.device
        )

        x = (
            self.token_emb(input_ids)
            + self.pos_emb(positions)[None, :, :]
        )

        for block in self.blocks:
            x = block(x)

        return self.final_norm(x)


# ============================================================
# LOAD CHECKPOINT STRICTLY
# ============================================================

print("Loading checkpoint...")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint step:", checkpoint["step"])
print("Checkpoint best loss:", checkpoint["best"])

model = Arch1()

load_result = model.load_state_dict(
    checkpoint["model"],
    strict=True
)

print("Strict loading:", load_result)

param_count = sum(
    p.numel() for p in model.parameters()
)

print(f"Model parameters: {param_count:,}")
print(f"Expected parameters: {EXPECTED_PARAMS:,}")
print("Parameter count matches:", param_count == EXPECTED_PARAMS)

print(
    "Embedding weights tied:",
    model.token_emb.weight.data_ptr()
    == model.lm_head.weight.data_ptr()
)

assert param_count == EXPECTED_PARAMS
assert (
    model.token_emb.weight.data_ptr()
    == model.lm_head.weight.data_ptr()
)

# ============================================================
# GPU FORWARD-PASS TEST
# ============================================================

model = model.to(DEVICE)
model.eval()

test_input = torch.tensor(
    [
        [1, 1410, 927, 893, 334, 263, 805, 2],
        [1, 1410, 1961, 1049, 263, 1061, 805, 2]
    ],
    dtype=torch.long,
    device=DEVICE
)

with torch.inference_mode():
    hidden = model(test_input)

print("\nForward-pass test:")
print("Input shape:", tuple(test_input.shape))
print("Hidden shape:", tuple(hidden.shape))
print("Device:", hidden.device)
print("Output finite:", torch.isfinite(hidden).all().item())

assert hidden.shape == (
    test_input.shape[0],
    test_input.shape[1],
    D_MODEL
)

assert torch.isfinite(hidden).all()

print("\nARCH1 SANITY CHECK PASSED")



Loading checkpoint...
Checkpoint step: 10000
Checkpoint best loss: 4.100625859101613
Strict loading: <All keys matched successfully>
Model parameters: 88,210,944
Expected parameters: 88,210,944
Parameter count matches: True
Embedding weights tied: True

Forward-pass test:
Input shape: (2, 8)
Hidden shape: (2, 8, 768)
Device: cuda:0
Output finite: True

ARCH1 SANITY CHECK PASSED


In [19]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer

# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
TRAIN_DIR = DATA_DIR / "Train"

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/achiver07/"
    "phase1-tokens-v40k-clean1/tokenizer.json"
)

MAX_LEN = 128
BATCH_SIZE = 2

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

NUM_LABELS = 13

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2

# ============================================================
# 1. LOAD TOKENIZER AND LABEL VOCABULARY
# ============================================================

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

with open(TRAIN_DIR / "label_vocab.json", "r") as f:
    label_vocab = json.load(f)

print("Vocabulary size:", tokenizer.get_vocab_size())
print("Number of labels:", len(label_vocab))
print("Labels:", label_vocab)

assert tokenizer.get_vocab_size() == 40000
assert len(label_vocab) == NUM_LABELS

# ============================================================
# 2. PREPARE SENTENCE-LEVEL DATASET
# ============================================================

class RhetoricalRoleDataset(Dataset):
    def __init__(self, file_paths, tokenizer, label_vocab, max_len=128):
        self.samples = []
        self.tokenizer = tokenizer
        self.label_vocab = label_vocab
        self.max_len = max_len

        for file_path in file_paths:
            with open(file_path, "r") as f:
                documents = json.load(f)

            print(
                f"Loading {file_path.name}: "
                f"{len(documents)} documents"
            )

            for doc in documents:
                sentences = doc["text"]
                labels = doc["labels"]

                assert len(sentences) == len(labels), (
                    f"Sentence-label mismatch in document {doc.get('id')}"
                )

                for sentence, label in zip(sentences, labels):
                    if label not in label_vocab:
                        raise ValueError(
                            f"Unknown label {label!r} "
                            f"in {file_path.name}"
                        )

                    self.samples.append((sentence, label_vocab[label]))

        print("Total sentence samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sentence, label = self.samples[idx]

        # Reserve two positions for BOS and EOS.
        token_ids = self.tokenizer.encode(sentence).ids
        token_ids = token_ids[:self.max_len - 2]

        input_ids = [BOS_ID] + token_ids + [EOS_ID]
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label
        }


def collate_batch(batch):
    max_length = max(
        len(item["input_ids"]) for item in batch
    )

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:
        padding_length = max_length - len(item["input_ids"])

        input_ids.append(
            item["input_ids"] + [PAD_ID] * padding_length
        )

        attention_masks.append(
            item["attention_mask"] + [0] * padding_length
        )

        labels.append(item["labels"])

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(
            attention_masks, dtype=torch.long
        ),
        "labels": torch.tensor(labels, dtype=torch.long)
    }


train_files = [
    TRAIN_DIR / "CL_train.json",
    TRAIN_DIR / "IT_train.json"
]

train_dataset = RhetoricalRoleDataset(
    train_files,
    tokenizer,
    label_vocab,
    max_len=MAX_LEN
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0
)

# ============================================================
# 3. CUSTOM LORA LINEAR LAYER
# ============================================================

class LoRALinear(nn.Module):
    def __init__(
        self,
        base_layer,
        rank=8,
        alpha=16,
        dropout=0.05
    ):
        super().__init__()

        if not isinstance(base_layer, nn.Linear):
            raise TypeError("LoRA target must be nn.Linear")

        self.base = base_layer
        self.rank = rank
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        # Freeze original pretrained linear layer.
        for param in self.base.parameters():
            param.requires_grad = False

        # Trainable low-rank matrices.
        self.lora_A = nn.Parameter(
            torch.empty(rank, in_features)
        )

        self.lora_B = nn.Parameter(
            torch.zeros(out_features, rank)
        )

        nn.init.kaiming_uniform_(
            self.lora_A,
            a=5**0.5
        )

    def forward(self, x):
        base_output = self.base(x)

        lora_output = F.linear(
            self.dropout(x),
            self.lora_A
        )

        lora_output = F.linear(
            lora_output,
            self.lora_B
        )

        return base_output + self.scaling * lora_output


# ============================================================
# 4. FREEZE BACKBONE AND INJECT LORA
# ============================================================

# Freeze every original Arch1 parameter.
for param in model.parameters():
    param.requires_grad = False

lora_modules = []

for layer_idx, block in enumerate(model.blocks):
    # Wrap query projection.
    block.attn.q_proj = LoRALinear(
        block.attn.q_proj,
        rank=LORA_RANK,
        alpha=LORA_ALPHA,
        dropout=LORA_DROPOUT
    )

    # Wrap value projection.
    block.attn.v_proj = LoRALinear(
        block.attn.v_proj,
        rank=LORA_RANK,
        alpha=LORA_ALPHA,
        dropout=LORA_DROPOUT
    )

    lora_modules.extend([
        f"blocks.{layer_idx}.attn.q_proj",
        f"blocks.{layer_idx}.attn.v_proj"
    ])

# ============================================================
# 5. CLASSIFICATION HEAD
# ============================================================

class RRClassifier(nn.Module):
    def __init__(self, backbone, d_model=768, num_labels=13):
        super().__init__()

        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(d_model, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(input_ids)

        # Use the last non-padding token representation.
        last_positions = attention_mask.sum(dim=1) - 1

        batch_indices = torch.arange(
            input_ids.shape[0],
            device=input_ids.device
        )

        pooled = hidden[batch_indices, last_positions]

        return self.classifier(pooled)


classifier_model = RRClassifier(
    backbone=model,
    d_model=768,
    num_labels=NUM_LABELS
).to(DEVICE)

# ============================================================
# 6. VERIFY TRAINABLE PARAMETERS
# ============================================================

trainable_params = [
    p for p in classifier_model.parameters()
    if p.requires_grad
]

trainable_count = sum(
    p.numel() for p in trainable_params
)

total_count = sum(
    p.numel() for p in classifier_model.parameters()
)

print("\nLoRA targets:", len(lora_modules))
print("Trainable parameters:", f"{trainable_count:,}")
print("Total parameters:", f"{total_count:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_count / total_count:.3f}%"
)

assert len(lora_modules) == 2 * N_LAYERS
assert trainable_count > 0

# ============================================================
# 7. SINGLE-BATCH FORWARD + BACKWARD TEST
# ============================================================

classifier_model.train()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=2e-4,
    weight_decay=0.01
)

optimizer.zero_grad(set_to_none=True)

logits = classifier_model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

loss = F.cross_entropy(logits, labels)

print("\nBatch test:")
print("Input shape:", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))
print("Labels:", labels.tolist())
print("Loss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

assert logits.shape == (input_ids.shape[0], NUM_LABELS)
assert torch.isfinite(loss)

loss.backward()

lora_gradients = [
    p.grad for name, p in classifier_model.named_parameters()
    if "lora_" in name and p.grad is not None
]

classifier_gradients = [
    p.grad for name, p in classifier_model.named_parameters()
    if "classifier" in name and p.grad is not None
]

print("LoRA gradient tensors:", len(lora_gradients))
print("Classifier gradient tensors:", len(classifier_gradients))

assert len(lora_gradients) > 0
assert len(classifier_gradients) > 0

lora_grad_norm = torch.sqrt(
    sum(g.detach().float().square().sum() for g in lora_gradients)
)

print("LoRA gradient norm:", lora_grad_norm.item())

assert torch.isfinite(lora_grad_norm)

print("\nSINGLE-BATCH FORWARD/BACKWARD TEST PASSED")
print("Full training has NOT started.")

Vocabulary size: 40000
Number of labels: 13
Labels: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}
Loading CL_train.json: 40 documents
Loading IT_train.json: 40 documents
Total sentence samples: 16845

LoRA targets: 16
Trainable parameters: 206,605
Total parameters: 88,417,549
Trainable percentage: 0.234%

Batch test:
Input shape: (2, 36)
Logits shape: (2, 13)
Labels: [0, 10]
Loss: 2.8655495643615723
Loss finite: True
LoRA gradient tensors: 32
Classifier gradient tensors: 2
LoRA gradient norm: 0.1070256158709526

SINGLE-BATCH FORWARD/BACKWARD TEST PASSED
Full training has NOT started.


In [20]:
import torch

if torch.cuda.is_available():
    torch.cuda.synchronize()

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )
    print(
        "Reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )
    print(
        "Free:",
        round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
        "GB"
    )

GPU: Tesla T4
Allocated: 0.38 GB
Reserved: 0.43 GB
Free: 13.99 GB


In [21]:
import json
import time
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 6
PATIENCE = 2

LR = 2e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

NUM_LABELS = 13

OUTPUT_DIR = Path(
    "/kaggle/working/phase3_rr_exp1_arch1_lora"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = (
    OUTPUT_DIR / "best_arch1_lora_rr.pt"
)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "CUDA GPU is required."

print("GPU:", torch.cuda.get_device_name(0))
print("Free GPU memory:",
      round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")

# ============================================================
# 2. BUILD VALIDATION DATASETS
# ============================================================

# Reuse the existing dataset implementation.
cl_dev_dataset = RhetoricalRoleDataset(
    [TRAIN_DIR / "CL_dev.json"],
    tokenizer,
    label_vocab,
    max_len=MAX_LEN
)

it_dev_dataset = RhetoricalRoleDataset(
    [TRAIN_DIR / "IT_dev.json"],
    tokenizer,
    label_vocab,
    max_len=MAX_LEN
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

cl_dev_loader = DataLoader(
    cl_dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

it_dev_loader = DataLoader(
    it_dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

print("\nTraining samples:", len(train_dataset))
print("CL validation samples:", len(cl_dev_dataset))
print("IT validation samples:", len(it_dev_dataset))
print("Training batches per epoch:", len(train_loader))

# ============================================================
# 3. CLASS WEIGHTS
# ============================================================

# Counts are calculated from the combined training dataset.
train_labels = torch.tensor(
    [label for _, label in train_dataset.samples],
    dtype=torch.long
)

class_counts = torch.bincount(
    train_labels,
    minlength=NUM_LABELS
).float()

print("\nTraining class counts:")
for label_name, label_id in sorted(
    label_vocab.items(), key=lambda item: item[1]
):
    print(
        f"{label_name:28s}: "
        f"{int(class_counts[label_id])}"
    )

# Square-root inverse-frequency weighting.
# Cap extreme weights for extremely rare labels.
class_weights = torch.sqrt(
    class_counts.sum()
    / (NUM_LABELS * class_counts.clamp_min(1))
)

class_weights = class_weights.clamp(max=3.0)

# Normalize so average class weight is approximately one.
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# ============================================================
# 4. OPTIMIZER AND AMP
# ============================================================

trainable_params = [
    p for p in classifier_model.parameters()
    if p.requires_grad
]

assert trainable_params, "No trainable parameters found."

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# FP16 autocast on CUDA; GradScaler protects small gradients.
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

print("\nTrainable parameters:",
      f"{sum(p.numel() for p in trainable_params):,}")

# ============================================================
# 5. VALIDATION FUNCTION
# ============================================================

ALL_LABEL_IDS = list(range(NUM_LABELS))


def evaluate(model, loader):
    model.eval()

    all_predictions = []
    all_targets = []

    total_loss = 0.0
    total_examples = 0

    with torch.inference_mode():
        for batch in loader:
            input_ids = batch["input_ids"].to(
                DEVICE, non_blocking=True
            )
            attention_mask = batch["attention_mask"].to(
                DEVICE, non_blocking=True
            )
            targets = batch["labels"].to(
                DEVICE, non_blocking=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                loss = criterion(logits, targets)

            predictions = logits.argmax(dim=-1)

            batch_size = targets.size(0)

            total_loss += loss.item() * batch_size
            total_examples += batch_size

            all_predictions.extend(
                predictions.cpu().tolist()
            )
            all_targets.extend(
                targets.cpu().tolist()
            )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        labels=ALL_LABEL_IDS,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_targets,
        all_predictions,
        labels=ALL_LABEL_IDS,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    return {
        "loss": total_loss / max(total_examples, 1),
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "accuracy": accuracy,
        "predictions": all_predictions,
        "targets": all_targets
    }


# ============================================================
# 6. SAVE ONLY TRAINABLE PARAMETERS
# ============================================================

def save_adapter_checkpoint(
    model,
    epoch,
    score,
    cl_metrics,
    it_metrics
):
    state = model.state_dict()

    # Keep LoRA adapters and classification head only.
    adapter_state = {
        key: value.detach().cpu().clone()
        for key, value in state.items()
        if (
            "lora_A" in key
            or "lora_B" in key
            or key.startswith("classifier.")
        )
    }

    if not adapter_state:
        raise RuntimeError(
            "No LoRA/classifier parameters found in checkpoint."
        )

    payload = {
        "epoch": epoch,
        "mean_validation_macro_f1": score,
        "cl_macro_f1": cl_metrics["macro_f1"],
        "it_macro_f1": it_metrics["macro_f1"],
        "adapter_state": adapter_state,
        "label_vocab": label_vocab,
        "config": {
            "base_architecture": "Arch1",
            "base_parameters": 88_210_944,
            "lora_rank": 8,
            "lora_alpha": 16,
            "target_modules": ["q_proj", "v_proj"],
            "max_length": MAX_LEN,
            "num_labels": NUM_LABELS
        }
    }

    torch.save(payload, BEST_CHECKPOINT)

    print("Saved best adapter checkpoint:", BEST_CHECKPOINT)
    print("Saved trainable tensors:", len(adapter_state))


# ============================================================
# 7. TRAINING LOOP
# ============================================================

best_score = -1.0
best_epoch = 0
epochs_without_improvement = 0

history = []

for epoch in range(1, EPOCHS + 1):

    classifier_model.train()

    epoch_start = time.time()

    running_loss = 0.0
    examples_seen = 0

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(
            DEVICE, non_blocking=True
        )
        attention_mask = batch["attention_mask"].to(
            DEVICE, non_blocking=True
        )
        targets = batch["labels"].to(
            DEVICE, non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = classifier_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, targets)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}, "
                f"batch {step}"
            )

        # Gradient accumulation.
        scaled_loss = loss / GRAD_ACCUM

        scaler.scale(scaled_loss).backward()

        should_update = (
            (step + 1) % GRAD_ACCUM == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                MAX_GRAD_NORM
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        batch_size = targets.size(0)

        running_loss += loss.item() * batch_size
        examples_seen += batch_size

        if (step + 1) % 100 == 0:
            print(
                f"Epoch {epoch}/{EPOCHS} | "
                f"Batch {step + 1}/{len(train_loader)} | "
                f"Loss {running_loss / examples_seen:.4f}"
            )

    train_loss = running_loss / max(examples_seen, 1)

    # --------------------------------------------------------
    # VALIDATE ON BOTH DOMAINS
    # --------------------------------------------------------

    cl_metrics = evaluate(
        classifier_model,
        cl_dev_loader
    )

    it_metrics = evaluate(
        classifier_model,
        it_dev_loader
    )

    mean_macro_f1 = (
        cl_metrics["macro_f1"]
        + it_metrics["macro_f1"]
    ) / 2.0

    elapsed = time.time() - epoch_start

    result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "cl_macro_f1": cl_metrics["macro_f1"],
        "it_macro_f1": it_metrics["macro_f1"],
        "mean_macro_f1": mean_macro_f1,
        "cl_weighted_f1": cl_metrics["weighted_f1"],
        "it_weighted_f1": it_metrics["weighted_f1"],
        "cl_accuracy": cl_metrics["accuracy"],
        "it_accuracy": it_metrics["accuracy"],
        "seconds": elapsed
    }

    history.append(result)

    print("\n" + "=" * 65)
    print(f"EPOCH {epoch}/{EPOCHS} COMPLETE")
    print("=" * 65)

    print(f"Training loss:       {train_loss:.4f}")

    print(
        f"CL | Macro-F1: {cl_metrics['macro_f1']:.4f} | "
        f"Weighted-F1: {cl_metrics['weighted_f1']:.4f} | "
        f"Accuracy: {cl_metrics['accuracy']:.4f}"
    )

    print(
        f"IT | Macro-F1: {it_metrics['macro_f1']:.4f} | "
        f"Weighted-F1: {it_metrics['weighted_f1']:.4f} | "
        f"Accuracy: {it_metrics['accuracy']:.4f}"
    )

    print(f"Mean validation Macro-F1: {mean_macro_f1:.4f}")
    print(f"Epoch time: {elapsed / 60:.2f} minutes")

    # --------------------------------------------------------
    # SAVE BEST CHECKPOINT
    # --------------------------------------------------------

    if mean_macro_f1 > best_score:
        best_score = mean_macro_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        save_adapter_checkpoint(
            classifier_model,
            epoch,
            best_score,
            cl_metrics,
            it_metrics
        )

        print("New best validation Macro-F1!")
    else:
        epochs_without_improvement += 1

        print(
            "No improvement:",
            epochs_without_improvement,
            "epoch(s)"
        )

    # Save training history after every epoch.
    with open(OUTPUT_DIR / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print("=" * 65 + "\n")

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n" + "#" * 65)
print("EXPERIMENT 1 COMPLETE")
print("#" * 65)

print("Best epoch:", best_epoch)
print(f"Best mean validation Macro-F1: {best_score:.4f}")
print("Best adapter:", BEST_CHECKPOINT)
print("History:", OUTPUT_DIR / "training_history.json")

GPU: Tesla T4
Free GPU memory: 13.99 GB
Loading CL_dev.json: 5 documents
Total sentence samples: 1396
Loading IT_dev.json: 5 documents
Total sentence samples: 746

Training samples: 16845
CL validation samples: 1396
IT validation samples: 746
Training batches per epoch: 2106

Training class counts:
Fact                        : 4098
Issue                       : 107
ArgumentPetitioner          : 1613
ArgumentRespondent          : 1348
Statute                     : 1099
Dissent                     : 284
PrecedentReliedUpon         : 1945
PrecedentNotReliedUpon      : 569
PrecedentOverruled          : 3
RulingByLowerCourt          : 874
RatioOfTheDecision          : 4301
RulingByPresentCourt        : 307
None                        : 297

Trainable parameters: 206,605
Epoch 1/6 | Batch 100/2106 | Loss 2.4101
Epoch 1/6 | Batch 200/2106 | Loss 2.3921
Epoch 1/6 | Batch 300/2106 | Loss 2.3806
Epoch 1/6 | Batch 400/2106 | Loss 2.3637
Epoch 1/6 | Batch 500/2106 | Loss 2.3452
Epoch 1/6 | Batch 

In [22]:
# ============================================================
# EXPERIMENT 1: DETAILED ERROR ANALYSIS
# Arch1 + Custom LoRA
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

# ------------------------------------------------------------
# 1. CONFIGURATION CHECK
# ------------------------------------------------------------

assert "label_vocab" in globals(), "label_vocab not found."
assert "cl_metrics" in globals(), "CL validation results not found."
assert "it_metrics" in globals(), "IT validation results not found."

label_names = [
    name
    for name, idx in sorted(
        label_vocab.items(),
        key=lambda item: item[1]
    )
]

label_ids = list(range(len(label_names)))

# ------------------------------------------------------------
# 2. PER-CLASS PERFORMANCE
# ------------------------------------------------------------

def analyze_domain(domain, metrics):
    y_true = np.asarray(metrics["targets"])
    y_pred = np.asarray(metrics["predictions"])

    assert len(y_true) == len(y_pred)
    assert len(y_true) > 0

    print("\n" + "=" * 85)
    print(f"{domain} VALIDATION RESULTS")
    print("=" * 85)

    report = classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=label_names,
        output_dict=True,
        zero_division=0
    )

    results = pd.DataFrame([
        {
            "Label": name,
            "Support": int(report[name]["support"]),
            "Precision": report[name]["precision"],
            "Recall": report[name]["recall"],
            "F1": report[name]["f1-score"]
        }
        for name in label_names
    ])

    results = results.sort_values(
        "F1",
        ascending=True
    ).reset_index(drop=True)

    print("\n--- Per-class results (worst performing first) ---")
    print(
        results.to_string(
            index=False,
            formatters={
                "Precision": "{:.3f}".format,
                "Recall": "{:.3f}".format,
                "F1": "{:.3f}".format
            }
        )
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        labels=label_ids,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(y_true, y_pred)

    print("\n--- Overall metrics ---")
    print(f"Macro-F1:   {macro_f1:.4f}")
    print(f"Weighted-F1: {weighted_f1:.4f}")
    print(f"Accuracy:   {accuracy:.4f}")

    # --------------------------------------------------------
    # 3. CONFUSION MATRIX
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=label_ids
    )

    cm_df = pd.DataFrame(
        cm,
        index=label_names,
        columns=label_names
    )

    print("\n--- Confusion matrix ---")
    print("Rows = actual labels; columns = predicted labels.")
    print(cm_df.to_string())

    # --------------------------------------------------------
    # 4. MOST COMMON MISCLASSIFICATIONS
    # --------------------------------------------------------

    errors = []

    for actual_id in label_ids:
        for predicted_id in label_ids:
            if actual_id == predicted_id:
                continue

            count = int(cm[actual_id, predicted_id])

            if count > 0:
                errors.append({
                    "Actual": label_names[actual_id],
                    "Predicted": label_names[predicted_id],
                    "Count": count
                })

    errors_df = pd.DataFrame(errors)

    print("\n--- Top 15 classification errors ---")

    if not errors_df.empty:
        errors_df = errors_df.sort_values(
            "Count",
            ascending=False
        ).head(15)

        print(errors_df.to_string(index=False))
    else:
        print("No classification errors.")

    # --------------------------------------------------------
    # 5. PREDICTION DISTRIBUTION
    # --------------------------------------------------------

    actual_counts = np.bincount(
        y_true,
        minlength=len(label_names)
    )

    predicted_counts = np.bincount(
        y_pred,
        minlength=len(label_names)
    )

    distribution = pd.DataFrame({
        "Label": label_names,
        "Actual_Count": actual_counts,
        "Predicted_Count": predicted_counts
    })

    print("\n--- Actual vs predicted class counts ---")
    print(distribution.to_string(index=False))

    return {
        "per_class": results,
        "confusion_matrix": cm_df,
        "top_errors": errors_df,
        "distribution": distribution,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "accuracy": accuracy
    }


# ------------------------------------------------------------
# 6. RUN ANALYSIS ON BOTH DOMAINS
# ------------------------------------------------------------

cl_analysis = analyze_domain("CL", cl_metrics)
it_analysis = analyze_domain("IT", it_metrics)

# ------------------------------------------------------------
# 7. COMPARE DOMAINS
# ------------------------------------------------------------

comparison = pd.DataFrame([
    {
        "Domain": "CL",
        "Macro-F1": cl_analysis["macro_f1"],
        "Weighted-F1": cl_analysis["weighted_f1"],
        "Accuracy": cl_analysis["accuracy"]
    },
    {
        "Domain": "IT",
        "Macro-F1": it_analysis["macro_f1"],
        "Weighted-F1": it_analysis["weighted_f1"],
        "Accuracy": it_analysis["accuracy"]
    }
])

print("\n" + "=" * 65)
print("FINAL DOMAIN COMPARISON")
print("=" * 65)
print(comparison.to_string(index=False))

# ------------------------------------------------------------
# 8. SAVE REPORTS
# ------------------------------------------------------------

OUTPUT_DIR = "/kaggle/working/phase3_rr_exp1_arch1_lora"

cl_analysis["per_class"].to_csv(
    f"{OUTPUT_DIR}/CL_per_class_results.csv",
    index=False
)

it_analysis["per_class"].to_csv(
    f"{OUTPUT_DIR}/IT_per_class_results.csv",
    index=False
)

cl_analysis["confusion_matrix"].to_csv(
    f"{OUTPUT_DIR}/CL_confusion_matrix.csv"
)

it_analysis["confusion_matrix"].to_csv(
    f"{OUTPUT_DIR}/IT_confusion_matrix.csv"
)

cl_analysis["top_errors"].to_csv(
    f"{OUTPUT_DIR}/CL_top_errors.csv",
    index=False
)

it_analysis["top_errors"].to_csv(
    f"{OUTPUT_DIR}/IT_top_errors.csv",
    index=False
)

print("\nReports saved in:", OUTPUT_DIR)
print("ERROR ANALYSIS COMPLETE")


CL VALIDATION RESULTS

--- Per-class results (worst performing first) ---
                 Label  Support Precision Recall    F1
                 Issue        3     0.000  0.000 0.000
               Dissent        0     0.000  0.000 0.000
PrecedentNotReliedUpon        0     0.000  0.000 0.000
    PrecedentOverruled        0     0.000  0.000 0.000
    RulingByLowerCourt       56     0.143  0.036 0.057
                  None        2     0.037  0.500 0.069
    ArgumentRespondent      236     0.321  0.106 0.159
   PrecedentReliedUpon      142     0.241  0.232 0.237
    ArgumentPetitioner      344     0.538  0.166 0.253
  RulingByPresentCourt       18     0.276  0.444 0.340
    RatioOfTheDecision      288     0.288  0.483 0.361
                  Fact      249     0.375  0.655 0.477
               Statute       58     0.661  0.638 0.649

--- Overall metrics ---
Macro-F1:   0.2002
Weighted-F1: 0.3067
Accuracy:   0.3331

--- Confusion matrix ---
Rows = actual labels; columns = predicted labe

In [23]:
import json
from pathlib import Path

CL_TEST_PATH = Path(
    "/kaggle/working/Phase_3_RR/Phase_3_RR/Test/CL_test.json"
)
IT_TEST_PATH = Path(
    "/kaggle/working/Phase_3_RR/Phase_3_RR/Test/IT_test.json"
)

for path in [CL_TEST_PATH, IT_TEST_PATH]:
    print("\n" + "=" * 60)
    print("FILE:", path.name)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Top-level type:", type(data).__name__)

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys())[:10])

        for key, value in list(data.items())[:2]:
            print(f"\nKey: {key!r}")
            print("Value type:", type(value).__name__)

            if isinstance(value, dict):
                print("Value keys:", list(value.keys())[:10])
                print("Sample:", str(value)[:1000])
            elif isinstance(value, list):
                print("List length:", len(value))
                print("First item:", str(value[:1])[:1000])
            else:
                print("Sample:", repr(value)[:500])

    elif isinstance(data, list):
        print("Number of documents:", len(data))
        print("First document:", str(data[:1])[:1500])

    else:
        print("Sample:", repr(data)[:1000])


FILE: CL_test.json
Top-level type: dict
Top-level keys: ['HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489', 'COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224', 'SC_Ultra Tech Cement Ltd vs Competition Commission of India and Ors 12062013 SC', 'CCI_Ms_Mineral_Enterprises_Limited_vs_Ministry_of_RailCO120089COM803307', 'HC_Vodafone_India_Limited_and_Ors_vs_The_Competition_MH2017051017164746274COM906883']

Key: 'HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489'
Value type: list
List length: 18
First item: ['Manmohan, J. 1 .']

Key: 'COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224'
Value type: list
List length: 230
First item: ['This appeal is directed against order dated 05.04.2016 passed by the Competition Commission of India (for short, the Commission), which has been termed by the appellant as an order passed under Section 26 of the Competition A

In [24]:
import json
from pathlib import Path

BASE_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
TEST_DIR = BASE_DIR / "Test"

def load_documents(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if not isinstance(raw, dict):
        raise ValueError(f"Expected dict in {path}, got {type(raw).__name__}")

    documents = []

    for doc_id, sentence_list in raw.items():
        sentences = []

        for i, item in enumerate(sentence_list):
            if isinstance(item, list) and len(item) == 1:
                sentence = item[0]
            elif isinstance(item, str):
                sentence = item
            else:
                raise ValueError(
                    f"Unexpected format: document={doc_id}, "
                    f"sentence={i}, item={item!r}"
                )

            if not isinstance(sentence, str):
                raise ValueError(f"Sentence is not a string: {doc_id}, {i}")

            sentences.append(sentence)

        documents.append({"id": doc_id, "text": sentences})

    return documents


cl_test_docs = load_documents(TEST_DIR / "CL_test.json")
it_test_docs = load_documents(TEST_DIR / "IT_test.json")

print("CL documents:", len(cl_test_docs))
print("IT documents:", len(it_test_docs))
print("CL sentences:", sum(len(d["text"]) for d in cl_test_docs))
print("IT sentences:", sum(len(d["text"]) for d in it_test_docs))

print("\nSample:")
print(cl_test_docs[0]["id"])
print(cl_test_docs[0]["text"][:2])

CL documents: 5
IT documents: 5
CL sentences: 1339
IT sentences: 858

Sample:
HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489
['Manmohan, J. 1 .', 'Present writ petition has been filed, challenging the notice dated 26th February, 2016 and respondent Number 1s order dated 18th February, 2016 whereby the respondent-CCI has directed the opposite parties in Reference Case Number 8/2013 to file their suggestions objections to the report of Director General (for short DG) and furnish their audited balance sheet, profit and loss account, turnover for the last three financial years.']


In [25]:
import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())

for name in ["classifier_model", "tokenizer", "label_vocab"]:
    print(f"{name} available:", name in globals())

adapter_path = Path(
    "/kaggle/working/phase3_rr_exp1_arch1_lora/best_arch1_lora_rr.pt"
)

print("LoRA adapter exists:", adapter_path.exists())

if adapter_path.exists():
    print("Adapter size:", round(adapter_path.stat().st_size / 1024**2, 2), "MB")

if "classifier_model" in globals():
    print("Model class:", type(classifier_model).__name__)
    print("Model device:", next(classifier_model.parameters()).device)

CUDA available: True
classifier_model available: True
tokenizer available: True
label_vocab available: True
LoRA adapter exists: True
Adapter size: 0.8 MB
Model class: RRClassifier
Model device: cuda:0


In [26]:
import inspect
import torch

print("Classifier class:", type(classifier_model).__name__)
print("\nForward signature:")
print(inspect.signature(classifier_model.forward))

print("\nTokenizer type:", type(tokenizer).__name__)
print("Tokenizer vocab size:", tokenizer.get_vocab_size())

print("\nLabel vocabulary:")
print(label_vocab)

print("\nModel training mode:", classifier_model.training)

Classifier class: RRClassifier

Forward signature:
(input_ids, attention_mask)

Tokenizer type: Tokenizer
Tokenizer vocab size: 40000

Label vocabulary:
{'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}

Model training mode: False


In [27]:
import json
import torch
from pathlib import Path
from torch.nn.utils.rnn import pad_sequence

# Paths
BASE_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
TEST_DIR = BASE_DIR / "Test"
OUTPUT_PATH = Path("/kaggle/working/prediction.json")

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
MAX_LEN = 128
BATCH_SIZE = 16

# Convert label vocabulary to ID -> label.
id_to_label = {int(v): k for k, v in label_vocab.items()}

def load_test_data(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if not isinstance(raw, dict):
        raise ValueError(f"Unexpected test format: {path}")

    documents = []

    for doc_id, sentence_list in raw.items():
        sentences = []

        for item in sentence_list:
            if isinstance(item, list) and len(item) == 1:
                sentence = item[0]
            elif isinstance(item, str):
                sentence = item
            else:
                raise ValueError(
                    f"Unexpected sentence format in {doc_id}: {item!r}"
                )

            sentences.append(sentence)

        documents.append({"id": doc_id, "text": sentences})

    return documents


cl_test_docs = load_test_data(TEST_DIR / "CL_test.json")
it_test_docs = load_test_data(TEST_DIR / "IT_test.json")

def encode_sentence(sentence):
    token_ids = tokenizer.encode(sentence).ids
    token_ids = token_ids[:MAX_LEN - 2]
    return [BOS_ID] + token_ids + [EOS_ID]


@torch.inference_mode()
def predict_sentences(sentences):
    classifier_model.eval()
    predictions = []

    for start in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[start:start + BATCH_SIZE]

        encoded = [
            torch.tensor(encode_sentence(s), dtype=torch.long)
            for s in batch
        ]

        input_ids = pad_sequence(
            encoded,
            batch_first=True,
            padding_value=PAD_ID
        ).to(DEVICE)

        attention_mask = (input_ids != PAD_ID).long()

        output = classifier_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if hasattr(output, "logits"):
            output = output.logits

        if isinstance(output, dict):
            output = output["logits"]

        if output.ndim != 2:
            raise ValueError(
                f"Expected [batch, labels] logits, got {tuple(output.shape)}"
            )

        predicted_ids = output.argmax(dim=-1).cpu().tolist()
        predictions.extend(id_to_label[i] for i in predicted_ids)

    assert len(predictions) == len(sentences)
    return predictions


# Predict all test documents.
prediction = {}

for split, documents in [
    ("CL", cl_test_docs),
    ("IT", it_test_docs)
]:
    print(f"Predicting {split}...")

    for index, doc in enumerate(documents, start=1):
        doc_id = doc["id"]

        if doc_id in prediction:
            raise ValueError(f"Duplicate document ID: {doc_id}")

        prediction[doc_id] = predict_sentences(doc["text"])

        assert len(prediction[doc_id]) == len(doc["text"])

        print(
            f"\r{split}: {index}/{len(documents)} documents",
            end=""
        )

    print()

# Save predictions.
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False, indent=2)

# Validate saved predictions.
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    saved = json.load(f)

expected_docs = cl_test_docs + it_test_docs

assert len(saved) == len(expected_docs)

for doc in expected_docs:
    doc_id = doc["id"]
    assert doc_id in saved
    assert len(saved[doc_id]) == len(doc["text"])
    assert all(label in label_vocab for label in saved[doc_id])

print("\nSUCCESS")
print("Output:", OUTPUT_PATH)
print("Documents:", len(saved))
print("Total predictions:", sum(len(v) for v in saved.values()))

Predicting CL...
CL: 5/5 documents
Predicting IT...
IT: 5/5 documents

SUCCESS
Output: /kaggle/working/prediction.json
Documents: 10
Total predictions: 2197


In [28]:
import json
import torch
from pathlib import Path
from torch.nn.utils.rnn import pad_sequence

# --------------------------------------------------
# Configuration
# --------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEST_DIR = Path("/kaggle/working/Phase_3_RR/Phase_3_RR/Test")
OUTPUT_PATH = Path("/kaggle/working/prediction.json")

MAX_LEN = 128
BATCH_SIZE = 16
PAD_ID, BOS_ID, EOS_ID = 0, 1, 2

id_to_label = {v: k for k, v in label_vocab.items()}

# --------------------------------------------------
# Load test documents
# --------------------------------------------------
def load_test_data(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if not isinstance(raw, dict):
        raise ValueError(f"Unexpected JSON structure: {path}")

    documents = []

    for doc_id, sentence_list in raw.items():
        sentences = []

        for item in sentence_list:
            if isinstance(item, list) and len(item) == 1:
                sentence = item[0]
            elif isinstance(item, str):
                sentence = item
            else:
                raise ValueError(
                    f"Unexpected sentence structure in {doc_id}: {item!r}"
                )

            sentences.append(sentence)

        documents.append({"id": doc_id, "text": sentences})

    return documents


cl_test_docs = load_test_data(TEST_DIR / "CL_test.json")
it_test_docs = load_test_data(TEST_DIR / "IT_test.json")

# --------------------------------------------------
# Tokenization
# --------------------------------------------------
def encode_sentence(sentence):
    ids = tokenizer.encode(sentence).ids
    ids = ids[:MAX_LEN - 2]
    return [BOS_ID] + ids + [EOS_ID]


# --------------------------------------------------
# Inference
# --------------------------------------------------
@torch.inference_mode()
def predict_sentences(sentences):
    predictions = []

    classifier_model.eval()

    for start in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[start:start + BATCH_SIZE]

        encoded = [
            torch.tensor(encode_sentence(s), dtype=torch.long)
            for s in batch
        ]

        input_ids = pad_sequence(
            encoded,
            batch_first=True,
            padding_value=PAD_ID
        ).to(DEVICE)

        attention_mask = (input_ids != PAD_ID).long()

        logits = classifier_model(input_ids, attention_mask)

        if hasattr(logits, "logits"):
            logits = logits.logits

        if isinstance(logits, dict):
            logits = logits["logits"]

        if logits.ndim != 2:
            raise ValueError(
                f"Expected logits [batch, labels], got {tuple(logits.shape)}"
            )

        if logits.shape[-1] != len(label_vocab):
            raise ValueError(
                f"Expected {len(label_vocab)} labels, got {logits.shape[-1]}"
            )

        predicted_ids = logits.argmax(dim=-1).cpu().tolist()
        predictions.extend(id_to_label[i] for i in predicted_ids)

    assert len(predictions) == len(sentences)
    return predictions


# --------------------------------------------------
# Predict every document
# --------------------------------------------------
prediction = {}

for split_name, documents in [
    ("CL", cl_test_docs),
    ("IT", it_test_docs),
]:
    print(f"\nPredicting {split_name}...")

    for index, doc in enumerate(documents, start=1):
        doc_id = doc["id"]

        if doc_id in prediction:
            raise ValueError(f"Duplicate document ID: {doc_id}")

        prediction[doc_id] = predict_sentences(doc["text"])

        assert len(prediction[doc_id]) == len(doc["text"])

        print(
            f"\r{split_name}: {index}/{len(documents)} documents",
            end=""
        )

    print()

# --------------------------------------------------
# Save and validate
# --------------------------------------------------
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False, indent=2)

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    saved = json.load(f)

expected_docs = cl_test_docs + it_test_docs

assert len(saved) == len(expected_docs)

for doc in expected_docs:
    doc_id = doc["id"]

    assert doc_id in saved
    assert len(saved[doc_id]) == len(doc["text"])
    assert all(label in label_vocab for label in saved[doc_id])

print("\n\nSUCCESS")
print("Prediction file:", OUTPUT_PATH)
print("Documents:", len(saved))
print("Total sentence predictions:", sum(len(v) for v in saved.values()))
print("\nSample:")
print(json.dumps(dict(list(saved.items())[:1]), indent=2, ensure_ascii=False))


Predicting CL...
CL: 5/5 documents

Predicting IT...
IT: 5/5 documents


SUCCESS
Prediction file: /kaggle/working/prediction.json
Documents: 10
Total sentence predictions: 2197

Sample:
{
  "HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489": [
    "None",
    "Fact",
    "Fact",
    "Fact",
    "ArgumentPetitioner",
    "ArgumentRespondent",
    "ArgumentRespondent",
    "RatioOfTheDecision",
    "ArgumentPetitioner",
    "Fact",
    "PrecedentReliedUpon",
    "ArgumentRespondent",
    "PrecedentReliedUpon",
    "RatioOfTheDecision",
    "PrecedentReliedUpon",
    "ArgumentRespondent",
    "RulingByPresentCourt",
    "RatioOfTheDecision"
  ]
}


In [29]:
import zipfile
from pathlib import Path

prediction_path = Path("/kaggle/working/prediction.json")
submission_path = Path("/kaggle/working/submission.zip")

assert prediction_path.exists(), "prediction.json does not exist"

with zipfile.ZipFile(
    submission_path, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(prediction_path, arcname="prediction.json")

with zipfile.ZipFile(submission_path, "r") as zf:
    assert zf.namelist() == ["prediction.json"]
    print("ZIP contents:", zf.namelist())

print("Submission ready:", submission_path)
print("Size:", round(submission_path.stat().st_size / 1024, 2), "KB")

ZIP contents: ['prediction.json']
Submission ready: /kaggle/working/submission.zip
Size: 2.37 KB


In [30]:
import pandas as pd
from pathlib import Path

RESULT_DIR = Path(
    "/kaggle/working/phase3_rr_exp1_arch1_lora"
)

CL_PATH = RESULT_DIR / "CL_per_class_results.csv"
IT_PATH = RESULT_DIR / "IT_per_class_results.csv"

print("CL results exist:", CL_PATH.exists())
print("IT results exist:", IT_PATH.exists())

if not CL_PATH.exists() or not IT_PATH.exists():
    print("\nAvailable files:")
    for path in RESULT_DIR.iterdir():
        print(path.name)
else:
    cl_df = pd.read_csv(CL_PATH)
    it_df = pd.read_csv(IT_PATH)

    print("\nCL result columns:")
    print(cl_df.columns.tolist())

    print("\nCL per-class results:")
    display(cl_df)

    print("\nIT per-class results:")
    display(it_df)

CL results exist: True
IT results exist: True

CL result columns:
['Label', 'Support', 'Precision', 'Recall', 'F1']

CL per-class results:


,Label,Support,Precision,Recall,F1
0,Issue,3,0.000000,0.000000,0.000000
1,Dissent,0,0.000000,0.000000,0.000000
2,PrecedentNotReliedUpon,0,0.000000,0.000000,0.000000
3,PrecedentOverruled,0,0.000000,0.000000,0.000000
4,RulingByLowerCourt,56,0.142857,0.035714,0.057143
5,NaN,2,0.037037,0.500000,0.068966
6,ArgumentRespondent,236,0.320513,0.105932,0.159236
7,PrecedentReliedUpon,142,0.240876,0.232394,0.236559
8,ArgumentPetitioner,344,0.537736,0.165698,0.253333
9,RulingByPresentCourt,18,0.275862,0.444444,0.340426



IT per-class results:


,Label,Support,Precision,Recall,F1
0,Dissent,0,0.000000,0.000000,0.000000
1,PrecedentOverruled,0,0.000000,0.000000,0.000000
2,PrecedentNotReliedUpon,50,0.175439,0.200000,0.186916
3,ArgumentPetitioner,56,0.263158,0.178571,0.212766
4,ArgumentRespondent,36,0.352941,0.166667,0.226415
5,RulingByLowerCourt,46,0.354839,0.239130,0.285714
6,RulingByPresentCourt,31,0.320000,0.258065,0.285714
7,PrecedentReliedUpon,112,0.356322,0.276786,0.311558
8,NaN,31,0.322581,0.322581,0.322581
9,RatioOfTheDecision,115,0.274590,0.582609,0.373259


In [31]:
import inspect
import torch
import torch.nn as nn

print("=" * 60)
print("1. MODEL INFORMATION")
print("=" * 60)

print("Model class:", type(classifier_model))
print("Model module:", type(classifier_model).__module__)
print("Constructor signature:")

try:
    print(inspect.signature(RRClassifier.__init__))
except Exception as e:
    print("Constructor source unavailable:", repr(e))

print("\n" + "=" * 60)
print("2. MODEL LAYERS")
print("=" * 60)

print(classifier_model)

print("\n" + "=" * 60)
print("3. PARAMETER INFORMATION")
print("=" * 60)

total_params = sum(p.numel() for p in classifier_model.parameters())
trainable_params = sum(
    p.numel()
    for p in classifier_model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\n" + "=" * 60)
print("4. FORWARD SIGNATURE")
print("=" * 60)

print(inspect.signature(classifier_model.forward))

print("\n" + "=" * 60)
print("5. MODEL CHILD MODULES")
print("=" * 60)

for name, module in classifier_model.named_children():
    print(f"{name}: {type(module).__name__}")

print("\n" + "=" * 60)
print("6. STATE DICT KEYS")
print("=" * 60)

for key in list(classifier_model.state_dict().keys())[:50]:
    print(key)

1. MODEL INFORMATION
Model class: <class '__main__.RRClassifier'>
Model module: __main__
Constructor signature:
(self, backbone, d_model=768, num_labels=13)

2. MODEL LAYERS
RRClassifier(
  (backbone): Arch1(
    (token_emb): Embedding(40000, 768)
    (pos_emb): Embedding(1024, 768)
    (blocks): ModuleList(
      (0-7): 8 x TransformerBlock(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (q_proj): LoRALinear(
            (base): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.05, inplace=False)
          )
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): LoRALinear(
            (base): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.05, inplace=False)
          )
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-05, e

In [32]:
import inspect
import torch

backbone = classifier_model.backbone
device = next(backbone.parameters()).device

print("Backbone:", type(backbone).__name__)

try:
    print("Forward signature:", inspect.signature(backbone.forward))
except Exception as e:
    print("Signature unavailable:", e)

dummy_ids = torch.tensor(
    [[1, 100, 200, 2, 0, 0]],
    dtype=torch.long,
    device=device
)

backbone.eval()

with torch.inference_mode():
    backbone_output = backbone(dummy_ids)

print("\nOutput type:", type(backbone_output).__name__)

if isinstance(backbone_output, torch.Tensor):
    print("Output shape:", tuple(backbone_output.shape))
elif isinstance(backbone_output, dict):
    print("Output keys:", list(backbone_output.keys()))
    for key, value in backbone_output.items():
        if isinstance(value, torch.Tensor):
            print(key, tuple(value.shape))
elif isinstance(backbone_output, (tuple, list)):
    print("Number of outputs:", len(backbone_output))
    for i, value in enumerate(backbone_output):
        if isinstance(value, torch.Tensor):
            print(f"Output {i} shape:", tuple(value.shape))
        else:
            print(f"Output {i} type:", type(value).__name__)

Backbone: Arch1
Forward signature: (input_ids)

Output type: Tensor
Output shape: (1, 6, 768)


In [33]:
import torch

classifier_model.eval()

device = next(classifier_model.parameters()).device

dummy_ids = torch.tensor(
    [[1, 100, 200, 2, 0, 0]],
    dtype=torch.long,
    device=device
)

dummy_mask = (dummy_ids != 0).long()

with torch.inference_mode():
    logits = classifier_model(dummy_ids, dummy_mask)

print("Classifier output shape:", tuple(logits.shape))
print("Output finite:", torch.isfinite(logits).all().item())
print("Predicted label ID:", logits.argmax(dim=-1).item())

Classifier output shape: (1, 13)
Output finite: True
Predicted label ID: 0


In [34]:
import torch
from pathlib import Path

adapter_path = Path(
    "/kaggle/working/phase3_rr_exp1_arch1_lora/"
    "best_arch1_lora_rr.pt"
)

assert adapter_path.exists(), f"Missing adapter: {adapter_path}"

checkpoint = torch.load(
    adapter_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    print(list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}: nested dictionary")
            print("Number of entries:", len(value))
            print("Sample keys:", list(value.keys())[:10])

        elif isinstance(value, torch.Tensor):
            print(f"{key}: shape={tuple(value.shape)}")

        else:
            print(f"{key}: {type(value).__name__}")

Checkpoint type: dict

Top-level keys:
['epoch', 'mean_validation_macro_f1', 'cl_macro_f1', 'it_macro_f1', 'adapter_state', 'label_vocab', 'config']
epoch: int
mean_validation_macro_f1: float
cl_macro_f1: float
it_macro_f1: float

adapter_state: nested dictionary
Number of entries: 34
Sample keys: ['backbone.blocks.0.attn.q_proj.lora_A', 'backbone.blocks.0.attn.q_proj.lora_B', 'backbone.blocks.0.attn.v_proj.lora_A', 'backbone.blocks.0.attn.v_proj.lora_B', 'backbone.blocks.1.attn.q_proj.lora_A', 'backbone.blocks.1.attn.q_proj.lora_B', 'backbone.blocks.1.attn.v_proj.lora_A', 'backbone.blocks.1.attn.v_proj.lora_B', 'backbone.blocks.2.attn.q_proj.lora_A', 'backbone.blocks.2.attn.q_proj.lora_B']

label_vocab: nested dictionary
Number of entries: 13
Sample keys: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt']

config: nested dictionary
Number of entries: 7
Sample k

In [35]:
import torch
from pathlib import Path

adapter_path = Path(
    "/kaggle/working/phase3_rr_exp1_arch1_lora/"
    "best_arch1_lora_rr.pt"
)

checkpoint = torch.load(
    adapter_path,
    map_location="cpu",
    weights_only=False
)

print("=" * 55)
print("BEST BASELINE CHECKPOINT")
print("=" * 55)

print("Best epoch:", checkpoint["epoch"])
print("Mean validation Macro-F1:",
      checkpoint["mean_validation_macro_f1"])
print("CL validation Macro-F1:", checkpoint["cl_macro_f1"])
print("IT validation Macro-F1:", checkpoint["it_macro_f1"])

print("\nConfiguration:")
for key, value in checkpoint["config"].items():
    print(f"{key}: {value}")

print("\nAdapter state contains classifier weights:")

for key in checkpoint["adapter_state"]:
    if "classifier" in key:
        print(key, checkpoint["adapter_state"][key].shape)

BEST BASELINE CHECKPOINT
Best epoch: 6
Mean validation Macro-F1: 0.2567081948047404
CL validation Macro-F1: 0.20018707718447898
IT validation Macro-F1: 0.3132293124250018

Configuration:
base_architecture: Arch1
base_parameters: 88210944
lora_rank: 8
lora_alpha: 16
target_modules: ['q_proj', 'v_proj']
max_length: 128
num_labels: 13

Adapter state contains classifier weights:
classifier.1.weight torch.Size([13, 768])
classifier.1.bias torch.Size([13])


In [36]:
import json
from pathlib import Path

TRAIN_DIR = Path(
    "/kaggle/working/Phase_3_RR/Phase_3_RR/Train"
)

files = [
    "CL_train.json",
    "CL_dev.json",
    "IT_train.json",
    "IT_dev.json",
]

for filename in files:
    path = TRAIN_DIR / filename

    print("\n" + "=" * 60)
    print(filename)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Top-level type:", type(data).__name__)

    if isinstance(data, dict):
        print("Number of top-level keys:", len(data))
        print("First keys:", list(data.keys())[:3])

        first_value = next(iter(data.values()), None)

        if isinstance(first_value, dict):
            print("Document keys:", list(first_value.keys()))

            for key, value in first_value.items():
                if isinstance(value, list):
                    print(f"{key}: length={len(value)}")
                    print("First item:", repr(value[:1])[:300])
                else:
                    print(f"{key}: {repr(value)[:150]}")

        elif isinstance(first_value, list):
            print("First value length:", len(first_value))
            print("First value item:", repr(first_value[:1])[:300])

    elif isinstance(data, list) and data:
        print("Number of documents:", len(data))
        print("First document:", repr(data[0])[:1000])


CL_train.json
Top-level type: list
Number of documents: 40
First document: {'id': 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733', 'text': ['Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the office of DGIR, MRTP Commission under Section 66(6) of the Competition Act, 2002.', 'The facts as stated in the complaint, in brief, are: 2 .1 This complaint was filed by the All India Distillers Association against glass bottle manufacturers in India alleging restrictive and monopolistic trade practices being adopted by the four major manufacturers namely: Haldyn Glass Gujarat Limited, Hindustan National Glass and Industries, AGI Glasspac and Mohan Crystal Glass Work who together control 90 of the glass bottle market.', '2.2 It has been stated that Potable Alcohol Industry of India comprises of Indian Made Foreign Liquor (IMFL), Country Liquor, Beer and Wine.', 'The packaging material for the potable alcohol is

In [37]:
# ============================================================
# PHASE 3 RR — ARCH1 + PREVIOUS-SENTENCE CONTEXT
# ============================================================

import copy
import json
import math
import random
import gc

from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, precision_recall_fscore_support


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BASE_DIR = Path(
    "/kaggle/working/Phase_3_RR/Phase_3_RR"
)

TRAIN_DIR = BASE_DIR / "Train"

OUTPUT_DIR = Path(
    "/kaggle/working/phase3_rr_exp2_arch1_context"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN = 256
CURRENT_MAX_TOKENS = 192
PREVIOUS_MAX_TOKENS = 61

BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 6

LR = 2e-4
WEIGHT_DECAY = 0.01

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2

NUM_LABELS = len(label_vocab)

LABELS = list(label_vocab.keys())
LABEL_IDS = list(range(NUM_LABELS))

ID_TO_LABEL = {
    label_id: label
    for label, label_id in label_vocab.items()
}

print("Device:", DEVICE)
print("Labels:", NUM_LABELS)
print("Output directory:", OUTPUT_DIR)


# ============================================================
# 2. CHECK EXISTING BASELINE
# ============================================================

assert "classifier_model" in globals()
assert "tokenizer" in globals()
assert "label_vocab" in globals()

baseline_model = classifier_model

baseline_model.eval()

print("\nBaseline model:", type(baseline_model).__name__)

print(
    "Baseline parameters:",
    f"{sum(p.numel() for p in baseline_model.parameters()):,}"
)

print(
    "Trainable baseline parameters:",
    f"{sum(p.numel() for p in baseline_model.parameters() if p.requires_grad):,}"
)


# ============================================================
# 3. LOAD TRAINING AND DEVELOPMENT DATA
# ============================================================

def load_documents(filename):

    path = TRAIN_DIR / filename

    with open(path, "r", encoding="utf-8") as f:
        documents = json.load(f)

    if not isinstance(documents, list):
        raise ValueError(
            f"{filename}: expected list, got {type(documents).__name__}"
        )

    for doc in documents:

        if "id" not in doc or "text" not in doc or "labels" not in doc:
            raise ValueError(
                f"{filename}: document must contain id, text and labels"
            )

        if len(doc["text"]) != len(doc["labels"]):
            raise ValueError(
                f"{filename}: sentence/label mismatch in {doc['id']}"
            )

    return documents


cl_train_docs = load_documents("CL_train.json")
it_train_docs = load_documents("IT_train.json")

cl_dev_docs = load_documents("CL_dev.json")
it_dev_docs = load_documents("IT_dev.json")

train_docs = cl_train_docs + it_train_docs

print("\nDocuments loaded:")
print("CL train:", len(cl_train_docs))
print("IT train:", len(it_train_docs))
print("CL dev:", len(cl_dev_docs))
print("IT dev:", len(it_dev_docs))


# ============================================================
# 4. NORMALIZE LABELS
# ============================================================

def normalize_label(label):

    # Handle JSON null as the "None" class if encountered.
    if label is None:
        label = "None"

    # Handle singleton-list label representation if present.
    if isinstance(label, list):
        if len(label) != 1:
            raise ValueError(f"Unexpected label format: {label!r}")
        label = label[0]

    if label is None:
        label = "None"

    if label not in label_vocab:
        raise ValueError(f"Unknown label: {label!r}")

    return label


for doc in train_docs + cl_dev_docs + it_dev_docs:

    doc["labels"] = [
        normalize_label(label)
        for label in doc["labels"]
    ]


# ============================================================
# 5. CLASS COUNTS AND CLASS WEIGHTS
# ============================================================

label_counts = Counter(
    label
    for doc in train_docs
    for label in doc["labels"]
)

print("\nTraining label counts:")

for label in LABELS:
    print(f"{label:28s} {label_counts.get(label, 0)}")


# Match the general strategy used in the baseline:
# square-root inverse-frequency weights, capped at 3,
# followed by normalization.

max_count = max(label_counts.values())

weights = []

for label in LABELS:

    count = label_counts.get(label, 0)

    if count == 0:
        weight = 1.0
    else:
        weight = min(
            3.0,
            math.sqrt(max_count / count)
        )

    weights.append(weight)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=DEVICE
)

class_weights = class_weights / class_weights.mean()

print("\nClass weights:", class_weights.detach().cpu().tolist())


# ============================================================
# 6. TOKENIZATION AND CONTEXT DATASET
# ============================================================

def encode_text(text):

    return tokenizer.encode(text).ids


class ContextRRDataset(Dataset):

    def __init__(self, documents):

        self.examples = []

        for doc in documents:

            sentences = doc["text"]
            labels = doc["labels"]

            assert len(sentences) == len(labels)

            previous_ids = []

            for sentence, label in zip(sentences, labels):

                current_ids = encode_text(sentence)

                # Preserve current-sentence information.
                current_ids = current_ids[:CURRENT_MAX_TOKENS]

                # Keep a limited amount of preceding context.
                prev_ids = previous_ids[-PREVIOUS_MAX_TOKENS:]

                # [BOS] previous [EOS] current [EOS]
                input_ids = (
                    [BOS_ID]
                    + prev_ids
                    + [EOS_ID]
                    + current_ids
                    + [EOS_ID]
                )

                # Index of the current sentence's final EOS.
                current_eos_position = len(input_ids) - 1

                if len(input_ids) > MAX_LEN:
                    raise ValueError(
                        f"Input exceeds MAX_LEN: {len(input_ids)}"
                    )

                self.examples.append({
                    "input_ids": input_ids,
                    "eos_position": current_eos_position,
                    "label_id": label_vocab[label],
                })

                # Context for the next sentence in this document.
                previous_ids = current_ids

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_context(batch):

    sequences = [
        torch.tensor(
            item["input_ids"],
            dtype=torch.long
        )
        for item in batch
    ]

    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=PAD_ID
    )

    attention_mask = (input_ids != PAD_ID).long()

    eos_positions = torch.tensor(
        [item["eos_position"] for item in batch],
        dtype=torch.long
    )

    labels = torch.tensor(
        [item["label_id"] for item in batch],
        dtype=torch.long
    )

    return input_ids, attention_mask, eos_positions, labels


train_dataset = ContextRRDataset(train_docs)

cl_dev_dataset = ContextRRDataset(cl_dev_docs)
it_dev_dataset = ContextRRDataset(it_dev_docs)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_context,
    num_workers=0
)

cl_dev_loader = DataLoader(
    cl_dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_context,
    num_workers=0
)

it_dev_loader = DataLoader(
    it_dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_context,
    num_workers=0
)

print("\nDataset sizes:")
print("Train sentences:", len(train_dataset))
print("CL-dev sentences:", len(cl_dev_dataset))
print("IT-dev sentences:", len(it_dev_dataset))


# ============================================================
# 7. PRESERVE BASELINE AND INITIALIZE CONTEXT MODEL
# ============================================================

context_model = copy.deepcopy(baseline_model)

context_model = context_model.to(DEVICE)

# Restore the saved best adapter explicitly when available.
adapter_path = Path(
    "/kaggle/working/phase3_rr_exp1_arch1_lora/"
    "best_arch1_lora_rr.pt"
)

if adapter_path.exists():

    checkpoint = torch.load(
        adapter_path,
        map_location="cpu",
        weights_only=False
    )

    if "adapter_state" in checkpoint:

        load_result = context_model.load_state_dict(
            checkpoint["adapter_state"],
            strict=False
        )

        print("\nLoaded baseline adapter state.")
        print("Unexpected keys:", load_result.unexpected_keys)

        # Missing base-model keys are expected because this checkpoint
        # contains only LoRA and classifier parameters.
        unexpected = load_result.unexpected_keys

        if unexpected:
            raise RuntimeError(
                f"Unexpected adapter keys: {unexpected}"
            )


# Freeze everything first.
for parameter in context_model.parameters():
    parameter.requires_grad = False

# Train LoRA matrices and the classification head.
for name, parameter in context_model.named_parameters():

    if (
        "lora_A" in name
        or "lora_B" in name
        or name.startswith("classifier.")
    ):
        parameter.requires_grad = True


trainable_parameters = [
    parameter
    for parameter in context_model.parameters()
    if parameter.requires_grad
]

print("\nContext model initialized.")

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in trainable_parameters):,}"
)


# ============================================================
# 8. FORWARD PASS USING CURRENT SENTENCE EOS
# ============================================================

def context_forward(model, input_ids, attention_mask, eos_positions):

    # Arch1.forward accepts input_ids only.
    hidden_states = model.backbone(input_ids)

    # Shape: [batch, sequence, hidden_dim]
    batch_indices = torch.arange(
        input_ids.size(0),
        device=input_ids.device
    )

    # Extract current sentence's EOS hidden state.
    pooled = hidden_states[
        batch_indices,
        eos_positions
    ]

    # Apply the existing classification head.
    logits = model.classifier(pooled)

    return logits


# Sanity check: forward pass and gradients.
context_model.eval()

sample_batch = next(iter(train_loader))

sample_ids, sample_mask, sample_eos, sample_labels = sample_batch

sample_ids = sample_ids.to(DEVICE)
sample_mask = sample_mask.to(DEVICE)
sample_eos = sample_eos.to(DEVICE)
sample_labels = sample_labels.to(DEVICE)

with torch.no_grad():

    sample_logits = context_forward(
        context_model,
        sample_ids,
        sample_mask,
        sample_eos
    )

print("\nForward sanity check:")
print("Input shape:", tuple(sample_ids.shape))
print("EOS positions:", sample_eos[:5].tolist())
print("Logits shape:", tuple(sample_logits.shape))

assert sample_logits.shape == (
    sample_ids.size(0),
    NUM_LABELS
)

assert torch.isfinite(sample_logits).all()

del sample_batch, sample_ids, sample_mask
del sample_eos, sample_labels, sample_logits

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# 9. EVALUATION
# ============================================================

def evaluate_model(model, loader):

    model.eval()

    all_true = []
    all_pred = []

    with torch.inference_mode():

        for input_ids, attention_mask, eos_positions, labels in loader:

            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            eos_positions = eos_positions.to(DEVICE)

            logits = context_forward(
                model,
                input_ids,
                attention_mask,
                eos_positions
            )

            predictions = logits.argmax(dim=-1).cpu().tolist()

            all_pred.extend(predictions)
            all_true.extend(labels.tolist())

    macro_f1 = f1_score(
        all_true,
        all_pred,
        labels=LABEL_IDS,
        average="macro",
        zero_division=0
    )

    return {
        "macro_f1": macro_f1,
        "y_true": all_true,
        "y_pred": all_pred
    }


# ============================================================
# 10. BASELINE EVALUATION ON THE SAME DEVELOPMENT DATA
# ============================================================

@torch.inference_mode()
def evaluate_sentence_baseline(model, documents):

    model.eval()

    y_true = []
    y_pred = []

    sentences = []
    labels = []

    for doc in documents:

        for sentence, label in zip(doc["text"], doc["labels"]):

            sentences.append(sentence)
            labels.append(label_vocab[label])

    for start in range(0, len(sentences), BATCH_SIZE):

        batch_sentences = sentences[start:start + BATCH_SIZE]

        encoded = []

        for sentence in batch_sentences:

            ids = encode_text(sentence)
            ids = ids[:MAX_LEN - 2]

            encoded.append(
                torch.tensor(
                    [BOS_ID] + ids + [EOS_ID],
                    dtype=torch.long
                )
            )

        input_ids = pad_sequence(
            encoded,
            batch_first=True,
            padding_value=PAD_ID
        ).to(DEVICE)

        attention_mask = (input_ids != PAD_ID).long()

        logits = model(input_ids, attention_mask)

        if isinstance(logits, dict):
            logits = logits["logits"]

        if hasattr(logits, "logits"):
            logits = logits.logits

        predicted = logits.argmax(dim=-1).cpu().tolist()

        y_pred.extend(predicted)

        y_true.extend(
            labels[start:start + len(batch_sentences)]
        )

    return {
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=LABEL_IDS,
            average="macro",
            zero_division=0
        ),
        "y_true": y_true,
        "y_pred": y_pred
    }


print("\nEvaluating original sentence-level baseline...")

baseline_cl = evaluate_sentence_baseline(
    baseline_model,
    cl_dev_docs
)

baseline_it = evaluate_sentence_baseline(
    baseline_model,
    it_dev_docs
)

baseline_combined_true = (
    baseline_cl["y_true"] + baseline_it["y_true"]
)

baseline_combined_pred = (
    baseline_cl["y_pred"] + baseline_it["y_pred"]
)

baseline_combined_f1 = f1_score(
    baseline_combined_true,
    baseline_combined_pred,
    labels=LABEL_IDS,
    average="macro",
    zero_division=0
)

print("Baseline CL Macro-F1:", round(baseline_cl["macro_f1"], 6))
print("Baseline IT Macro-F1:", round(baseline_it["macro_f1"], 6))
print("Baseline combined Macro-F1:", round(baseline_combined_f1, 6))


# ============================================================
# 11. OPTIMIZER AND LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

use_amp = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)


# ============================================================
# 12. TRAINING LOOP
# ============================================================

best_combined_f1 = -1.0
best_epoch = 0

history = []

best_path = OUTPUT_DIR / "best_arch1_context.pt"

for epoch in range(1, EPOCHS + 1):

    context_model.train()

    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0

    for step, batch in enumerate(train_loader, start=1):

        input_ids, attention_mask, eos_positions, labels = batch

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        eos_positions = eos_positions.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp
        ):

            logits = context_forward(
                context_model,
                input_ids,
                attention_mask,
                eos_positions
            )

            loss = criterion(logits, labels)

            # Gradient accumulation.
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        running_loss += loss.item() * GRAD_ACCUM

        if step % GRAD_ACCUM == 0 or step == len(train_loader):

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        if step % 200 == 0 or step == len(train_loader):

            print(
                f"\rEpoch {epoch}/{EPOCHS} "
                f"Step {step}/{len(train_loader)} "
                f"Loss {running_loss / step:.4f}",
                end=""
            )

    print()

    # Evaluate on development datasets.
    cl_result = evaluate_model(
        context_model,
        cl_dev_loader
    )

    it_result = evaluate_model(
        context_model,
        it_dev_loader
    )

    combined_true = (
        cl_result["y_true"] + it_result["y_true"]
    )

    combined_pred = (
        cl_result["y_pred"] + it_result["y_pred"]
    )

    combined_f1 = f1_score(
        combined_true,
        combined_pred,
        labels=LABEL_IDS,
        average="macro",
        zero_division=0
    )

    epoch_loss = running_loss / len(train_loader)

    result = {
        "epoch": epoch,
        "train_loss": epoch_loss,
        "cl_macro_f1": cl_result["macro_f1"],
        "it_macro_f1": it_result["macro_f1"],
        "combined_macro_f1": combined_f1
    }

    history.append(result)

    print(
        f"Epoch {epoch}: "
        f"loss={epoch_loss:.4f} | "
        f"CL={cl_result['macro_f1']:.4f} | "
        f"IT={it_result['macro_f1']:.4f} | "
        f"Combined={combined_f1:.4f}"
    )

    if combined_f1 > best_combined_f1:

        best_combined_f1 = combined_f1
        best_epoch = epoch

        trainable_state = {
            name: parameter.detach().cpu().clone()
            for name, parameter in context_model.named_parameters()
            if parameter.requires_grad
        }

        torch.save(
            {
                "epoch": epoch,
                "combined_macro_f1": combined_f1,
                "cl_macro_f1": cl_result["macro_f1"],
                "it_macro_f1": it_result["macro_f1"],
                "trainable_state": trainable_state,
                "label_vocab": label_vocab,
                "config": {
                    "base_architecture": "Arch1",
                    "lora_rank": 8,
                    "lora_alpha": 16,
                    "max_length": MAX_LEN,
                    "previous_sentence_context": True,
                    "current_eos_pooling": True,
                    "num_labels": NUM_LABELS
                }
            },
            best_path
        )

        print("Saved new best checkpoint:", best_path)


# ============================================================
# 13. SAVE TRAINING HISTORY
# ============================================================

with open(
    OUTPUT_DIR / "training_history.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(history, f, indent=2)


print("\n" + "=" * 65)
print("CONTEXT EXPERIMENT COMPLETE")
print("=" * 65)

print("Baseline combined dev Macro-F1:", round(baseline_combined_f1, 6))
print("Best context combined dev Macro-F1:", round(best_combined_f1, 6))
print("Best epoch:", best_epoch)
print("Best checkpoint:", best_path)

print(
    "Absolute development improvement:",
    round(best_combined_f1 - baseline_combined_f1, 6)
)

Device: cuda
Labels: 13
Output directory: /kaggle/working/phase3_rr_exp2_arch1_context

Baseline model: RRClassifier
Baseline parameters: 88,417,549
Trainable baseline parameters: 206,605

Documents loaded:
CL train: 40
IT train: 40
CL dev: 5
IT dev: 5

Training label counts:
Fact                         4098
Issue                        107
ArgumentPetitioner           1613
ArgumentRespondent           1348
Statute                      1099
Dissent                      284
PrecedentReliedUpon          1945
PrecedentNotReliedUpon       569
PrecedentOverruled           3
RulingByLowerCourt           874
RatioOfTheDecision           4301
RulingByPresentCourt         307
None                         297

Class weights: [0.46120643615722656, 1.3505724668502808, 0.7351300716400146, 0.80414879322052, 0.8905996084213257, 1.3505724668502808, 0.6694552302360535, 1.2377279996871948, 1.3505724668502808, 0.9986783862113953, 0.45019081234931946, 1.3505724668502808, 1.3505724668502808]

Dataset size

In [38]:
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/working/phase3_rr_exp2_arch1_context/"
    "best_arch1_context.pt"
)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Best epoch:", checkpoint["epoch"])
print("Combined development Macro-F1:",
      checkpoint["combined_macro_f1"])
print("CL development Macro-F1:",
      checkpoint["cl_macro_f1"])
print("IT development Macro-F1:",
      checkpoint["it_macro_f1"])

print("\nConfiguration:")
print(checkpoint["config"])

print("\nTrainable state entries:",
      len(checkpoint["trainable_state"]))

print("Sample state keys:")
for key in list(checkpoint["trainable_state"].keys())[:10]:
    print(key, tuple(checkpoint["trainable_state"][key].shape))

Best epoch: 3
Combined development Macro-F1: 0.27338321941208205
CL development Macro-F1: 0.21261156394007236
IT development Macro-F1: 0.29829019211359864

Configuration:
{'base_architecture': 'Arch1', 'lora_rank': 8, 'lora_alpha': 16, 'max_length': 256, 'previous_sentence_context': True, 'current_eos_pooling': True, 'num_labels': 13}

Trainable state entries: 34
Sample state keys:
backbone.blocks.0.attn.q_proj.lora_A (8, 768)
backbone.blocks.0.attn.q_proj.lora_B (768, 8)
backbone.blocks.0.attn.v_proj.lora_A (8, 768)
backbone.blocks.0.attn.v_proj.lora_B (768, 8)
backbone.blocks.1.attn.q_proj.lora_A (8, 768)
backbone.blocks.1.attn.q_proj.lora_B (768, 8)
backbone.blocks.1.attn.v_proj.lora_A (8, 768)
backbone.blocks.1.attn.v_proj.lora_B (768, 8)
backbone.blocks.2.attn.q_proj.lora_A (8, 768)
backbone.blocks.2.attn.q_proj.lora_B (768, 8)


In [39]:
import copy
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASELINE_ADAPTER_PATH = Path(
    "/kaggle/working/phase3_rr_exp1_arch1_lora/best_arch1_lora_rr.pt"
)
CONTEXT_CHECKPOINT_PATH = Path(
    "/kaggle/working/phase3_rr_exp2_arch1_context/best_arch1_context.pt"
)

assert BASELINE_ADAPTER_PATH.exists(), f"Missing: {BASELINE_ADAPTER_PATH}"
assert CONTEXT_CHECKPOINT_PATH.exists(), f"Missing: {CONTEXT_CHECKPOINT_PATH}"

# Load the best context checkpoint
context_ckpt = torch.load(
    CONTEXT_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

# Load the baseline adapter checkpoint
baseline_ckpt = torch.load(
    BASELINE_ADAPTER_PATH,
    map_location="cpu",
    weights_only=False
)

# Use the existing model architecture and create a clean copy
# classifier_model must be the initialized Arch1 + LoRA + RRClassifier model
context_infer_model = copy.deepcopy(classifier_model)

# First load baseline adapter weights
load_result = context_infer_model.load_state_dict(
    baseline_ckpt["adapter_state"],
    strict=False
)

print("Baseline adapter loaded.")
print("Unexpected keys:", load_result.unexpected_keys)

# Then overwrite trainable parameters with the BEST context checkpoint
named_params = dict(context_infer_model.named_parameters())
context_state = context_ckpt["trainable_state"]

missing_params = []
shape_errors = []

with torch.no_grad():
    for name, value in context_state.items():
        if name not in named_params:
            missing_params.append(name)
            continue

        param = named_params[name]

        if param.shape != value.shape:
            shape_errors.append(
                (name, tuple(param.shape), tuple(value.shape))
            )
            continue

        param.copy_(value.to(device=param.device, dtype=param.dtype))

assert not missing_params, f"Parameters not found: {missing_params}"
assert not shape_errors, f"Parameter shape mismatch: {shape_errors}"

context_infer_model = context_infer_model.to(DEVICE)
context_infer_model.eval()

print("\nContext model restored successfully.")
print("Best epoch:", context_ckpt["epoch"])
print("Development Macro-F1:", context_ckpt["combined_macro_f1"])
print("Trainable parameters restored:", len(context_state))
print("Device:", DEVICE)

Baseline adapter loaded.
Unexpected keys: []

Context model restored successfully.
Best epoch: 3
Development Macro-F1: 0.27338321941208205
Trainable parameters restored: 34
Device: cuda


In [40]:
import json
import zipfile
from pathlib import Path
from collections import Counter

import torch

# --------------------------------------------------
# PATHS AND CONFIGURATION
# --------------------------------------------------
ROOT = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
TEST_DIR = ROOT / "Test"

CL_TEST_PATH = TEST_DIR / "CL_test.json"
IT_TEST_PATH = TEST_DIR / "IT_test.json"

OUTPUT_DIR = Path("/kaggle/working/phase3_rr_exp2_arch1_context")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = OUTPUT_DIR / "prediction.json"
SUBMISSION_PATH = Path("/kaggle/working/submission.zip")

BOS_ID = 1
EOS_ID = 2
PAD_ID = 0

MAX_LEN = 256
CURRENT_MAX_TOKENS = 192
PREVIOUS_MAX_TOKENS = 61
INFER_BATCH_SIZE = 8

# Use the label vocabulary saved with the checkpoint
label_vocab = context_ckpt["label_vocab"]
id_to_label = {int(v): k for k, v in label_vocab.items()}

# --------------------------------------------------
# LOAD TEST DOCUMENTS
# --------------------------------------------------
def load_test_data(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    documents = []

    for doc_id, items in raw.items():
        sentences = []

        for item in items:
            if isinstance(item, str):
                sentences.append(item)
            elif isinstance(item, list):
                if len(item) != 1 or not isinstance(item[0], str):
                    raise ValueError(
                        f"Unexpected sentence structure in {doc_id}: {item}"
                    )
                sentences.append(item[0])
            else:
                raise TypeError(
                    f"Unexpected test item type in {doc_id}: {type(item)}"
                )

        documents.append({
            "id": doc_id,
            "text": sentences
        })

    return documents


cl_test_docs = load_test_data(CL_TEST_PATH)
it_test_docs = load_test_data(IT_TEST_PATH)

print("CL test documents:", len(cl_test_docs))
print("IT test documents:", len(it_test_docs))
print(
    "Total test sentences:",
    sum(len(d["text"]) for d in cl_test_docs + it_test_docs)
)

# --------------------------------------------------
# TOKENIZATION
# --------------------------------------------------
def encode_sentence(sentence):
    # Assumes the existing `tokenizer` is the Phase-1 tokenizer.
    return tokenizer.encode(sentence).ids[:CURRENT_MAX_TOKENS]


# --------------------------------------------------
# CONTEXT-AWARE INFERENCE
# --------------------------------------------------
@torch.inference_mode()
def predict_document(doc):
    sentences = doc["text"]
    predictions = []

    # Reset context at the start of EVERY document.
    previous_ids = []

    for start in range(0, len(sentences), INFER_BATCH_SIZE):
        batch_items = []

        for sentence in sentences[start:start + INFER_BATCH_SIZE]:
            current_ids = encode_sentence(sentence)
            prev_ids = previous_ids[-PREVIOUS_MAX_TOKENS:]

            # Match the training input format:
            # BOS + previous sentence + EOS + current sentence + EOS
            input_ids = (
                [BOS_ID]
                + prev_ids
                + [EOS_ID]
                + current_ids
                + [EOS_ID]
            )

            if len(input_ids) > MAX_LEN:
                raise ValueError(
                    f"Input length {len(input_ids)} exceeds {MAX_LEN}"
                )

            eos_position = len(input_ids) - 1

            batch_items.append({
                "input_ids": input_ids,
                "eos_position": eos_position,
                "current_ids": current_ids
            })

            # Update context for the NEXT sentence in this document.
            previous_ids = current_ids

        max_batch_len = max(
            len(item["input_ids"]) for item in batch_items
        )

        input_batch = []
        for item in batch_items:
            ids = item["input_ids"]
            padded = ids + [PAD_ID] * (max_batch_len - len(ids))
            input_batch.append(padded)

        input_ids = torch.tensor(
            input_batch,
            dtype=torch.long,
            device=DEVICE
        )

        eos_positions = torch.tensor(
            [item["eos_position"] for item in batch_items],
            dtype=torch.long,
            device=DEVICE
        )

        # Arch1 forward accepts input_ids only.
        hidden = context_infer_model.backbone(input_ids)

        batch_indices = torch.arange(
            input_ids.size(0), device=DEVICE
        )

        pooled = hidden[batch_indices, eos_positions]
        logits = context_infer_model.classifier(pooled)

        predicted_ids = logits.argmax(dim=-1).cpu().tolist()
        predictions.extend(id_to_label[int(i)] for i in predicted_ids)

    assert len(predictions) == len(sentences), (
        f"Prediction count mismatch for {doc['id']}: "
        f"{len(predictions)} predictions vs {len(sentences)} sentences"
    )

    return predictions


# --------------------------------------------------
# PREDICT CL + IT
# --------------------------------------------------
prediction = {}

for domain_name, documents in [
    ("CL", cl_test_docs),
    ("IT", it_test_docs)
]:
    print(f"\nPredicting {domain_name}...")

    for i, doc in enumerate(documents, start=1):
        prediction[doc["id"]] = predict_document(doc)

        if i % 1 == 0:
            print(
                f"  {i}/{len(documents)} documents | "
                f"{doc['id']} | "
                f"{len(prediction[doc['id']])} sentences"
            )

# --------------------------------------------------
# VALIDATE OUTPUT
# --------------------------------------------------
expected_doc_ids = {
    doc["id"] for doc in cl_test_docs + it_test_docs
}

assert set(prediction.keys()) == expected_doc_ids, (
    "Document IDs do not match the test documents."
)

expected_sentence_count = sum(
    len(doc["text"]) for doc in cl_test_docs + it_test_docs
)

actual_sentence_count = sum(
    len(labels) for labels in prediction.values()
)

assert actual_sentence_count == expected_sentence_count, (
    f"Expected {expected_sentence_count} predictions, "
    f"got {actual_sentence_count}"
)

allowed_labels = set(label_vocab.keys())

for doc_id, labels in prediction.items():
    invalid = set(labels) - allowed_labels
    assert not invalid, f"Invalid labels for {doc_id}: {invalid}"

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False)

# --------------------------------------------------
# CREATE SUBMISSION ZIP
# --------------------------------------------------
with zipfile.ZipFile(
    SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(PREDICTION_PATH, arcname="prediction.json")

# Verify ZIP contents
with zipfile.ZipFile(SUBMISSION_PATH, "r") as zf:
    names = zf.namelist()
    assert names == ["prediction.json"], (
        f"Unexpected ZIP contents: {names}"
    )

    check = json.loads(zf.read("prediction.json").decode("utf-8"))
    assert len(check) == len(expected_doc_ids)

print("\n" + "=" * 55)
print("SUBMISSION CREATED SUCCESSFULLY")
print("=" * 55)
print("Documents:", len(prediction))
print("Sentences:", actual_sentence_count)
print("Prediction JSON:", PREDICTION_PATH)
print("Submission ZIP:", SUBMISSION_PATH)
print("ZIP contents:", names)
print("\nPrediction label distribution:")
print(Counter(
    label
    for labels in prediction.values()
    for label in labels
))

CL test documents: 5
IT test documents: 5
Total test sentences: 2197

Predicting CL...
  1/5 documents | HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489 | 18 sentences
  2/5 documents | COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224 | 230 sentences
  3/5 documents | SC_Ultra Tech Cement Ltd vs Competition Commission of India and Ors 12062013 SC | 35 sentences
  4/5 documents | CCI_Ms_Mineral_Enterprises_Limited_vs_Ministry_of_RailCO120089COM803307 | 114 sentences
  5/5 documents | HC_Vodafone_India_Limited_and_Ors_vs_The_Competition_MH2017051017164746274COM906883 | 942 sentences

Predicting IT...
  1/5 documents | SC_2009_710 | 420 sentences
  2/5 documents | SC_2017_195 | 186 sentences
  3/5 documents | BHC_BomHC_2014_498 | 53 sentences
  4/5 documents | BHC_BomHC_1981_263 | 40 sentences
  5/5 documents | BHC_BomHC_1997_583 | 159 sentences

SUBMISSION CREATED SUCCESSFULLY
Documents: 10
Sentences: 2197
Prediction 

In [41]:
import copy
import json
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import f1_score

# ============================================================
# 1. CONFIGURATION
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
TRAIN_DIR = ROOT / "Train"

BASELINE_PATH = Path(
    "/kaggle/working/phase3_rr_exp1_arch1_lora/"
    "best_arch1_lora_rr.pt"
)

CONTEXT_PATH = Path(
    "/kaggle/working/phase3_rr_exp2_arch1_context/"
    "best_arch1_context.pt"
)

assert BASELINE_PATH.exists(), f"Baseline checkpoint missing: {BASELINE_PATH}"
assert CONTEXT_PATH.exists(), f"Context checkpoint missing: {CONTEXT_PATH}"

assert "classifier_model" in globals(), (
    "classifier_model is missing. Initialize the original Arch1 + LoRA "
    "classifier before running this cell."
)
assert "tokenizer" in globals(), "Phase-1 tokenizer is not available."

baseline_ckpt = torch.load(
    BASELINE_PATH,
    map_location="cpu",
    weights_only=False
)

context_ckpt = torch.load(
    CONTEXT_PATH,
    map_location="cpu",
    weights_only=False
)

label_vocab = baseline_ckpt["label_vocab"]

# Normalize label names for consistent evaluation.
label_to_id = {str(label): int(idx) for label, idx in label_vocab.items()}
id_to_label = {idx: label for label, idx in label_to_id.items()}

NUM_LABELS = len(label_to_id)

print("Device:", DEVICE)
print("Number of labels:", NUM_LABELS)
print("Context checkpoint best epoch:", context_ckpt["epoch"])
print("Context development Macro-F1:", context_ckpt["combined_macro_f1"])


# ============================================================
# 2. RESTORE BASELINE MODEL
# ============================================================

baseline_model = copy.deepcopy(classifier_model)

baseline_load = baseline_model.load_state_dict(
    baseline_ckpt["adapter_state"],
    strict=False
)

assert not baseline_load.unexpected_keys, (
    f"Unexpected baseline keys: {baseline_load.unexpected_keys}"
)

baseline_model = baseline_model.to(DEVICE)
baseline_model.eval()

print("\nBaseline model restored.")


# ============================================================
# 3. RESTORE BEST CONTEXT MODEL
# ============================================================

context_model = copy.deepcopy(baseline_model)

named_params = dict(context_model.named_parameters())

with torch.no_grad():
    for name, value in context_ckpt["trainable_state"].items():
        if name not in named_params:
            raise KeyError(f"Context parameter not found: {name}")

        param = named_params[name]

        if tuple(param.shape) != tuple(value.shape):
            raise ValueError(
                f"Shape mismatch for {name}: "
                f"{tuple(param.shape)} vs {tuple(value.shape)}"
            )

        param.copy_(
            value.to(device=param.device, dtype=param.dtype)
        )

context_model = context_model.to(DEVICE)
context_model.eval()

print("Best context model restored.")


# ============================================================
# 4. LOAD DEVELOPMENT DOCUMENTS
# ============================================================

def load_documents(path):
    with open(path, "r", encoding="utf-8") as f:
        documents = json.load(f)

    assert isinstance(documents, list), (
        f"Expected a list of documents in {path}"
    )

    return documents


dev_sets = {
    "CL": load_documents(TRAIN_DIR / "CL_dev.json"),
    "IT": load_documents(TRAIN_DIR / "IT_dev.json"),
}

for domain, docs in dev_sets.items():
    sentence_count = sum(len(doc["text"]) for doc in docs)
    print(
        f"{domain}: {len(docs)} documents, "
        f"{sentence_count} sentences"
    )


# ============================================================
# 5. TOKENIZATION AND BATCHING HELPERS
# ============================================================

BOS_ID = 1
EOS_ID = 2
PAD_ID = 0

BASE_MAX_LEN = 128
BASE_TOKEN_LIMIT = BASE_MAX_LEN - 2

CONTEXT_MAX_LEN = 256
CURRENT_MAX_TOKENS = 192
PREVIOUS_MAX_TOKENS = 61

BATCH_SIZE = 16


def encode_text(text, limit):
    return tokenizer.encode(str(text)).ids[:limit]


def make_padded_batch(sequences):
    max_len = max(len(x) for x in sequences)

    input_ids = [
        seq + [PAD_ID] * (max_len - len(seq))
        for seq in sequences
    ]

    input_ids = torch.tensor(
        input_ids,
        dtype=torch.long,
        device=DEVICE
    )

    attention_mask = (input_ids != PAD_ID).long()

    return input_ids, attention_mask


# ============================================================
# 6. BASELINE PROBABILITIES
# ============================================================

@torch.inference_mode()
def baseline_document_probs(sentences):
    all_probs = []

    sequences = []

    for sentence in sentences:
        token_ids = encode_text(sentence, BASE_TOKEN_LIMIT)

        # Sentence-only baseline input.
        seq = [BOS_ID] + token_ids + [EOS_ID]
        sequences.append(seq)

    for start in range(0, len(sequences), BATCH_SIZE):
        batch_sequences = sequences[start:start + BATCH_SIZE]

        input_ids, attention_mask = make_padded_batch(
            batch_sequences
        )

        logits = baseline_model(
            input_ids,
            attention_mask
        )

        probs = torch.softmax(logits.float(), dim=-1)

        all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs, axis=0)


# ============================================================
# 7. CONTEXT MODEL PROBABILITIES
# ============================================================

@torch.inference_mode()
def context_document_probs(sentences):
    sequences = []
    eos_positions = []

    previous_ids = []

    # Construct inputs in document order.
    for sentence in sentences:
        current_ids = encode_text(
            sentence,
            CURRENT_MAX_TOKENS
        )

        prev_ids = previous_ids[-PREVIOUS_MAX_TOKENS:]

        seq = (
            [BOS_ID]
            + prev_ids
            + [EOS_ID]
            + current_ids
            + [EOS_ID]
        )

        if len(seq) > CONTEXT_MAX_LEN:
            raise ValueError(
                f"Context input exceeds maximum length: {len(seq)}"
            )

        sequences.append(seq)
        eos_positions.append(len(seq) - 1)

        # Match the context training procedure.
        previous_ids = current_ids

    all_probs = []

    for start in range(0, len(sequences), BATCH_SIZE):
        batch_sequences = sequences[start:start + BATCH_SIZE]
        batch_positions = eos_positions[start:start + BATCH_SIZE]

        input_ids, _ = make_padded_batch(batch_sequences)

        positions = torch.tensor(
            batch_positions,
            dtype=torch.long,
            device=DEVICE
        )

        # Arch1 accepts input_ids only.
        hidden = context_model.backbone(input_ids)

        batch_indices = torch.arange(
            input_ids.size(0),
            device=DEVICE
        )

        pooled = hidden[batch_indices, positions]

        logits = context_model.classifier(pooled)

        probs = torch.softmax(logits.float(), dim=-1)

        all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs, axis=0)


# ============================================================
# 8. GENERATE DEVELOPMENT PROBABILITIES
# ============================================================

domain_results = {}

for domain, docs in dev_sets.items():
    print(f"\nGenerating {domain} development predictions...")

    domain_y_true = []
    domain_baseline_probs = []
    domain_context_probs = []

    for doc_idx, doc in enumerate(docs, start=1):
        sentences = doc["text"]
        labels = doc["labels"]

        if len(sentences) != len(labels):
            raise ValueError(
                f"Sentence/label mismatch in {domain}, doc {doc['id']}"
            )

        normalized_labels = [
            "None" if label is None else str(label)
            for label in labels
        ]

        unknown_labels = [
            label for label in normalized_labels
            if label not in label_to_id
        ]

        if unknown_labels:
            raise ValueError(
                f"Unknown labels in {doc['id']}: {unknown_labels[:5]}"
            )

        y_true = [label_to_id[label] for label in normalized_labels]

        base_probs = baseline_document_probs(sentences)
        ctx_probs = context_document_probs(sentences)

        assert len(base_probs) == len(y_true)
        assert len(ctx_probs) == len(y_true)

        domain_y_true.extend(y_true)
        domain_baseline_probs.append(base_probs)
        domain_context_probs.append(ctx_probs)

        print(
            f"  {doc_idx}/{len(docs)} | {doc['id']} | "
            f"{len(sentences)} sentences"
        )

    domain_results[domain] = {
        "y_true": np.asarray(domain_y_true, dtype=np.int64),
        "baseline_probs": np.concatenate(
            domain_baseline_probs, axis=0
        ),
        "context_probs": np.concatenate(
            domain_context_probs, axis=0
        ),
    }

print("\nDevelopment probabilities generated.")


# ============================================================
# 9. SWEEP ENSEMBLE WEIGHTS
# ============================================================

# alpha=0.0 means baseline only.
# alpha=1.0 means context model only.
ALPHAS = np.round(np.arange(0.0, 1.01, 0.1), 2)

all_y_true = np.concatenate([
    domain_results[d]["y_true"]
    for d in ["CL", "IT"]
])

all_base_probs = np.concatenate([
    domain_results[d]["baseline_probs"]
    for d in ["CL", "IT"]
])

all_context_probs = np.concatenate([
    domain_results[d]["context_probs"]
    for d in ["CL", "IT"]
])

LABEL_IDS = list(range(NUM_LABELS))

rows = []

for alpha in ALPHAS:
    # Probability ensemble:
    # (1 - alpha) * baseline + alpha * context
    ensemble_probs = (
        (1.0 - alpha) * all_base_probs
        + alpha * all_context_probs
    )

    combined_preds = ensemble_probs.argmax(axis=1)

    combined_f1 = f1_score(
        all_y_true,
        combined_preds,
        labels=LABEL_IDS,
        average="macro",
        zero_division=0
    )

    row = {
        "context_weight": float(alpha),
        "combined_macro_f1": float(combined_f1),
    }

    for domain in ["CL", "IT"]:
        y_true = domain_results[domain]["y_true"]

        base_probs = domain_results[domain]["baseline_probs"]
        ctx_probs = domain_results[domain]["context_probs"]

        domain_probs = (
            (1.0 - alpha) * base_probs
            + alpha * ctx_probs
        )

        preds = domain_probs.argmax(axis=1)

        row[f"{domain}_macro_f1"] = float(
            f1_score(
                y_true,
                preds,
                labels=LABEL_IDS,
                average="macro",
                zero_division=0
            )
        )

    rows.append(row)


# ============================================================
# 10. DISPLAY AND SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(rows).sort_values(
    "context_weight"
).reset_index(drop=True)

print("\n" + "=" * 75)
print("BASELINE + CONTEXT ENSEMBLE RESULTS")
print("=" * 75)

print(
    results_df.to_string(
        index=False,
        formatters={
            "context_weight": "{:.1f}".format,
            "combined_macro_f1": "{:.6f}".format,
            "CL_macro_f1": "{:.6f}".format,
            "IT_macro_f1": "{:.6f}".format,
        }
    )
)

# Choose the best pooled development Macro-F1.
best_idx = results_df["combined_macro_f1"].idxmax()
best_row = results_df.loc[best_idx]

baseline_row = results_df[
    results_df["context_weight"] == 0.0
].iloc[0]

improvement = (
    best_row["combined_macro_f1"]
    - baseline_row["combined_macro_f1"]
)

OUTPUT_DIR = Path(
    "/kaggle/working/phase3_rr_exp3_ensemble"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_path = OUTPUT_DIR / "ensemble_dev_results.csv"
results_df.to_csv(results_path, index=False)

summary = {
    "best_context_weight": float(best_row["context_weight"]),
    "best_combined_dev_macro_f1": float(
        best_row["combined_macro_f1"]
    ),
    "best_cl_dev_macro_f1": float(best_row["CL_macro_f1"]),
    "best_it_dev_macro_f1": float(best_row["IT_macro_f1"]),
    "baseline_only_dev_macro_f1": float(
        baseline_row["combined_macro_f1"]
    ),
    "improvement_over_baseline_dev": float(improvement),
}

with open(OUTPUT_DIR / "best_ensemble_config.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 60)
print("BEST ENSEMBLE CONFIGURATION")
print("=" * 60)
print(json.dumps(summary, indent=2))
print("\nSaved results:", results_path)

Device: cuda
Number of labels: 13
Context checkpoint best epoch: 3
Context development Macro-F1: 0.27338321941208205

Baseline model restored.
Best context model restored.
CL: 5 documents, 1396 sentences
IT: 5 documents, 746 sentences

Generating CL development predictions...
  1/5 | CCI_Noida_Software_Technology_Park_Ltd_vs_Star_India_PCO201808081816041524COM760948 | 265 sentences
  2/5 | SC_Competition Commission of India vs Bharti Airtel Limited and Ors 05122018 SC(1) | 848 sentences
  3/5 | COM_Shree_Cement_Limited__vs__Builders_Association_of_TA2015171215164519311COM615014 | 32 sentences
  4/5 | HC_State_of_Mizoram_vs_Competition_Commission_of_IndiGH201424121422183488COM227556 | 184 sentences
  5/5 | CCI_Western_Coalfield_Limited__vs__SSV_Coal_Carriers_PCO20152807152158246COM767401 | 67 sentences

Generating IT development predictions...
  1/5 | SC_2009_1357 | 76 sentences
  2/5 | KHC_KolHC_1980_69 | 120 sentences
  3/5 | BHC_BomHC_2014_1227 | 123 sentences
  4/5 | SC_2007_389 | 3

In [42]:
import json
import zipfile
import numpy as np
import torch

from pathlib import Path
from collections import Counter

# ============================================================
# CONFIGURATION
# ============================================================

ROOT = Path("/kaggle/working/Phase_3_RR/Phase_3_RR")
TEST_DIR = ROOT / "Test"

ALPHA = 0.7  # Selected using development Macro-F1

BOS_ID = 1
EOS_ID = 2
PAD_ID = 0

BASE_MAX_LEN = 128
BASE_TOKEN_LIMIT = BASE_MAX_LEN - 2

CONTEXT_MAX_LEN = 256
CURRENT_MAX_TOKENS = 192
PREVIOUS_MAX_TOKENS = 61

BATCH_SIZE = 16

OUTPUT_DIR = Path(
    "/kaggle/working/phase3_rr_exp3_ensemble"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = OUTPUT_DIR / "prediction.json"
SUBMISSION_PATH = OUTPUT_DIR / "submission.zip"

CL_TEST_PATH = TEST_DIR / "CL_test.json"
IT_TEST_PATH = TEST_DIR / "IT_test.json"

assert 0.0 <= ALPHA <= 1.0
assert CL_TEST_PATH.exists(), f"Missing {CL_TEST_PATH}"
assert IT_TEST_PATH.exists(), f"Missing {IT_TEST_PATH}"

baseline_model.eval()
context_model.eval()

print("Ensemble context weight:", ALPHA)
print("Baseline weight:", 1.0 - ALPHA)
print("Device:", DEVICE)


# ============================================================
# LOAD TEST DOCUMENTS
# ============================================================

def load_test_documents(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    documents = []

    for doc_id, items in raw.items():
        sentences = []

        for item in items:
            if isinstance(item, str):
                sentences.append(item)
            elif isinstance(item, list) and len(item) == 1:
                sentences.append(item[0])
            else:
                raise ValueError(
                    f"Unexpected sentence format in {doc_id}: {item}"
                )

        documents.append({
            "id": doc_id,
            "text": sentences
        })

    return documents


cl_test_docs = load_test_documents(CL_TEST_PATH)
it_test_docs = load_test_documents(IT_TEST_PATH)

all_test_docs = cl_test_docs + it_test_docs

print("CL documents:", len(cl_test_docs))
print("IT documents:", len(it_test_docs))
print(
    "Total test sentences:",
    sum(len(d["text"]) for d in all_test_docs)
)


# ============================================================
# TOKENIZATION AND PADDING
# ============================================================

def encode_text(text, limit):
    return tokenizer.encode(str(text)).ids[:limit]


def make_padded_batch(sequences):
    max_len = max(len(seq) for seq in sequences)

    padded = [
        seq + [PAD_ID] * (max_len - len(seq))
        for seq in sequences
    ]

    input_ids = torch.tensor(
        padded,
        dtype=torch.long,
        device=DEVICE
    )

    attention_mask = (input_ids != PAD_ID).long()

    return input_ids, attention_mask


# ============================================================
# BASELINE PROBABILITIES
# ============================================================

@torch.inference_mode()
def predict_baseline_probs(sentences):
    sequences = []

    for sentence in sentences:
        token_ids = encode_text(
            sentence,
            BASE_TOKEN_LIMIT
        )

        sequences.append(
            [BOS_ID] + token_ids + [EOS_ID]
        )

    outputs = []

    for start in range(0, len(sequences), BATCH_SIZE):
        batch = sequences[start:start + BATCH_SIZE]

        input_ids, attention_mask = make_padded_batch(batch)

        logits = baseline_model(
            input_ids,
            attention_mask
        )

        probs = torch.softmax(logits.float(), dim=-1)

        outputs.append(probs.cpu().numpy())

    return np.concatenate(outputs, axis=0)


# ============================================================
# CONTEXT MODEL PROBABILITIES
# ============================================================

@torch.inference_mode()
def predict_context_probs(sentences):
    sequences = []
    eos_positions = []

    previous_ids = []

    for sentence in sentences:
        current_ids = encode_text(
            sentence,
            CURRENT_MAX_TOKENS
        )

        prev_ids = previous_ids[-PREVIOUS_MAX_TOKENS:]

        sequence = (
            [BOS_ID]
            + prev_ids
            + [EOS_ID]
            + current_ids
            + [EOS_ID]
        )

        assert len(sequence) <= CONTEXT_MAX_LEN

        sequences.append(sequence)
        eos_positions.append(len(sequence) - 1)

        # Preserve the training-time context construction.
        previous_ids = current_ids

    outputs = []

    for start in range(0, len(sequences), BATCH_SIZE):
        batch = sequences[start:start + BATCH_SIZE]
        positions = eos_positions[start:start + BATCH_SIZE]

        input_ids, _ = make_padded_batch(batch)

        positions = torch.tensor(
            positions,
            dtype=torch.long,
            device=DEVICE
        )

        hidden = context_model.backbone(input_ids)

        batch_indices = torch.arange(
            input_ids.size(0),
            device=DEVICE
        )

        pooled = hidden[batch_indices, positions]
        logits = context_model.classifier(pooled)

        probs = torch.softmax(logits.float(), dim=-1)

        outputs.append(probs.cpu().numpy())

    return np.concatenate(outputs, axis=0)


# ============================================================
# GENERATE ENSEMBLE PREDICTIONS
# ============================================================

prediction = {}

for domain, documents in [
    ("CL", cl_test_docs),
    ("IT", it_test_docs)
]:
    print(f"\nProcessing {domain}...")

    for i, doc in enumerate(documents, start=1):
        sentences = doc["text"]

        base_probs = predict_baseline_probs(sentences)
        context_probs = predict_context_probs(sentences)

        assert base_probs.shape == context_probs.shape
        assert base_probs.shape[0] == len(sentences)

        # Weighted probability ensemble.
        ensemble_probs = (
            (1.0 - ALPHA) * base_probs
            + ALPHA * context_probs
        )

        predicted_ids = ensemble_probs.argmax(axis=1)

        predicted_labels = [
            id_to_label[int(idx)]
            for idx in predicted_ids
        ]

        prediction[doc["id"]] = predicted_labels

        print(
            f"  {i}/{len(documents)} | {doc['id']} | "
            f"{len(predicted_labels)} sentences"
        )


# ============================================================
# VALIDATE PREDICTIONS
# ============================================================

expected_ids = {doc["id"] for doc in all_test_docs}

assert set(prediction.keys()) == expected_ids, (
    "Prediction document IDs do not match test document IDs."
)

expected_count = sum(
    len(doc["text"]) for doc in all_test_docs
)

actual_count = sum(
    len(labels) for labels in prediction.values()
)

assert actual_count == expected_count, (
    f"Expected {expected_count} predictions; got {actual_count}"
)

allowed_labels = set(label_to_id.keys())

for doc_id, labels in prediction.items():
    assert all(label in allowed_labels for label in labels), (
        f"Invalid prediction label in {doc_id}"
    )

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False)


# ============================================================
# CREATE SUBMISSION ZIP
# ============================================================

with zipfile.ZipFile(
    SUBMISSION_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(
        PREDICTION_PATH,
        arcname="prediction.json"
    )

with zipfile.ZipFile(SUBMISSION_PATH, "r") as zf:
    assert zf.namelist() == ["prediction.json"]

    saved_predictions = json.loads(
        zf.read("prediction.json").decode("utf-8")
    )

assert saved_predictions == prediction

print("\n" + "=" * 60)
print("ENSEMBLE SUBMISSION READY")
print("=" * 60)
print("Context weight:", ALPHA)
print("Baseline weight:", 1.0 - ALPHA)
print("Documents:", len(prediction))
print("Sentences:", actual_count)
print("Label distribution:")

print(Counter(
    label
    for labels in prediction.values()
    for label in labels
))

print("\nPrediction JSON:", PREDICTION_PATH)
print("Submission ZIP:", SUBMISSION_PATH)
print("ZIP contents: prediction.json")

Ensemble context weight: 0.7
Baseline weight: 0.30000000000000004
Device: cuda
CL documents: 5
IT documents: 5
Total test sentences: 2197

Processing CL...
  1/5 | HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489 | 18 sentences
  2/5 | COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224 | 230 sentences
  3/5 | SC_Ultra Tech Cement Ltd vs Competition Commission of India and Ors 12062013 SC | 35 sentences
  4/5 | CCI_Ms_Mineral_Enterprises_Limited_vs_Ministry_of_RailCO120089COM803307 | 114 sentences
  5/5 | HC_Vodafone_India_Limited_and_Ors_vs_The_Competition_MH2017051017164746274COM906883 | 942 sentences

Processing IT...
  1/5 | SC_2009_710 | 420 sentences
  2/5 | SC_2017_195 | 186 sentences
  3/5 | BHC_BomHC_2014_498 | 53 sentences
  4/5 | BHC_BomHC_1981_263 | 40 sentences
  5/5 | BHC_BomHC_1997_583 | 159 sentences

ENSEMBLE SUBMISSION READY
Context weight: 0.7
Baseline weight: 0.30000000000000004
Documents: 10
Sentenc

In [43]:
from pathlib import Path

# Root directory for all Phase 3 submissions
SUBMISSIONS_ROOT = Path("/kaggle/working/Phase3_Submissions")

# Unique directory for this technique
TECHNIQUE_NAME = "03_baseline_context_ensemble"

OUTPUT_DIR = SUBMISSIONS_ROOT / TECHNIQUE_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = OUTPUT_DIR / "prediction.json"
SUBMISSION_PATH = OUTPUT_DIR / "submission.zip"
RESULTS_PATH = OUTPUT_DIR / "results.json"

print("Technique:", TECHNIQUE_NAME)
print("Output directory:", OUTPUT_DIR)
print("Prediction file:", PREDICTION_PATH)
print("Submission ZIP:", SUBMISSION_PATH)

Technique: 03_baseline_context_ensemble
Output directory: /kaggle/working/Phase3_Submissions/03_baseline_context_ensemble
Prediction file: /kaggle/working/Phase3_Submissions/03_baseline_context_ensemble/prediction.json
Submission ZIP: /kaggle/working/Phase3_Submissions/03_baseline_context_ensemble/submission.zip


In [44]:
import json
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path(
    "/kaggle/working/Phase3_Submissions/"
    "03_baseline_context_ensemble"
)

RESULTS_PATH = OUTPUT_DIR / "results.json"

results = {
    "technique": "Baseline + Context Ensemble",
    "context_weight": 0.7,
    "baseline_weight": 0.3,
    "development_macro_f1": 0.2911882984508824,
    "codabench_macro_f1": None,
    "status": "Awaiting Codabench evaluation",
    "submitted_at": None
}

# Codabench score milne ke baad values update karna.
# Example:
# results["codabench_macro_f1"] = 0.31
# results["status"] = "Evaluated"
# results["submitted_at"] = datetime.now().isoformat()

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Results record saved:", RESULTS_PATH)

Results record saved: /kaggle/working/Phase3_Submissions/03_baseline_context_ensemble/results.json


In [45]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/kaggle/working/Phase3_Submissions")

EXPERIMENTS = {
    "01_arch1_lora_baseline": {
        "technique": "Arch1 LoRA Baseline",
        "development_macro_f1": None,
        "codabench_macro_f1": 0.300542,
        "status": "Evaluated",
        "source_dir": Path(
            "/kaggle/working/phase3_rr_exp1_arch1_lora"
        ),
    },
    "02_arch1_previous_sentence_context": {
        "technique": "Arch1 Previous-Sentence Context",
        "development_macro_f1": 0.280618,
        "codabench_macro_f1": 0.277608,
        "status": "Evaluated",
        "source_dir": Path(
            "/kaggle/working/phase3_rr_exp2_arch1_context"
        ),
    },
    "03_baseline_context_ensemble": {
        "technique": "Baseline + Context Ensemble",
        "development_macro_f1": 0.291188,
        "codabench_macro_f1": None,
        "status": "Awaiting Codabench evaluation",
        "source_dir": Path(
            "/kaggle/working/phase3_rr_exp3_ensemble"
        ),
    },
}

ROOT.mkdir(parents=True, exist_ok=True)

leaderboard = []

for folder_name, info in EXPERIMENTS.items():
    output_dir = ROOT / folder_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # Copy existing experiment artifacts without overwriting files.
    source_dir = info["source_dir"]

    if source_dir.exists():
        for source_file in source_dir.iterdir():
            if source_file.is_file():
                destination = output_dir / source_file.name

                if not destination.exists():
                    destination.write_bytes(source_file.read_bytes())

    # Create a results record only if one doesn't already exist.
    results_path = output_dir / "results.json"

    if not results_path.exists():
        results = {
            "technique": info["technique"],
            "development_macro_f1": info["development_macro_f1"],
            "codabench_macro_f1": info["codabench_macro_f1"],
            "status": info["status"],
        }

        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)

    with open(results_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    leaderboard.append({
        "folder": folder_name,
        "technique": results.get("technique", info["technique"]),
        "development_macro_f1": results.get("development_macro_f1"),
        "codabench_macro_f1": results.get("codabench_macro_f1"),
        "status": results.get("status"),
    })

    print(f"\n{folder_name}")
    for file in sorted(output_dir.iterdir()):
        print("  ", file.name)

# Save a common leaderboard.
leaderboard_path = ROOT / "submission_leaderboard.csv"
pd.DataFrame(leaderboard).to_csv(leaderboard_path, index=False)

print("\n" + "=" * 60)
print("EXPERIMENT FILE ORGANIZATION COMPLETE")
print("=" * 60)
print("Root:", ROOT)
print("Leaderboard:", leaderboard_path)


01_arch1_lora_baseline
   CL_confusion_matrix.csv
   CL_per_class_results.csv
   CL_top_errors.csv
   IT_confusion_matrix.csv
   IT_per_class_results.csv
   IT_top_errors.csv
   best_arch1_lora_rr.pt
   results.json
   training_history.json

02_arch1_previous_sentence_context
   best_arch1_context.pt
   prediction.json
   results.json
   training_history.json

03_baseline_context_ensemble
   best_ensemble_config.json
   ensemble_dev_results.csv
   prediction.json
   results.json
   submission.zip

EXPERIMENT FILE ORGANIZATION COMPLETE
Root: /kaggle/working/Phase3_Submissions
Leaderboard: /kaggle/working/Phase3_Submissions/submission_leaderboard.csv


In [46]:
from pathlib import Path
import zipfile

SUBMISSION_PATH = Path(
    "/kaggle/working/Phase3_Submissions/"
    "03_baseline_context_ensemble/submission.zip"
)

print("ZIP exists:", SUBMISSION_PATH.exists())

if SUBMISSION_PATH.exists():
    print("File size:", round(SUBMISSION_PATH.stat().st_size / 1024, 2), "KB")

    with zipfile.ZipFile(SUBMISSION_PATH, "r") as zf:
        print("ZIP contents:", zf.namelist())
        assert zf.namelist() == ["prediction.json"]

    print("Submission ZIP is valid.")
else:
    print("ZIP not found. Regenerate the ensemble submission first.")

ZIP exists: True
File size: 2.09 KB
ZIP contents: ['prediction.json']
Submission ZIP is valid.


In [47]:
from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/Phase3_Submissions/"
    "03_baseline_context_ensemble/submission.zip"
))

/kaggle/working/Phase3_Submissions/03_baseline_context_ensemble/submission.zip